In [ ]:
!pip install -q transformers xgboost shap openpyxl plotly streamlit requests datasets google-generativeai
print("Packages installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.5 MB/s eta 0:00:00
Packages installed


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, torch, pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/crypto_signals'

COIN_NAME = 'ETH'
WORK_DIR = '/content/eth_final'

PRICE_CSV = f'{DRIVE_DIR}/Ethereum_Price_Data.csv'
SENT_FILE = f'{DRIVE_DIR}/Ethereum_NLp_Data.csv'
NEWS_CSV = f'{DRIVE_DIR}/coindesk_news_2020_2025.csv'
BINANCE_SYMBOL = 'ETHUSDT'
COIN_SUPPLY = 120_000_000

for sub in ['', '/outputs', '/outputs/sequences', '/outputs/models',
            '/outputs/evaluation', '/outputs/onchain', '/outputs/news']:
    os.makedirs(WORK_DIR + sub, exist_ok=True)
os.chdir(WORK_DIR)

for cache in ['_finbert_cache.csv', '_cryptobert_cache.csv', '_news_finbert_cache.csv']:
    dp = f"{DRIVE_DIR}/{cache}"
    lp = f"{WORK_DIR}/outputs/{cache}"
    if os.path.exists(dp) and not os.path.exists(lp):
        shutil.copy(dp, lp)
        print(f"  Restored {cache} ({len(pd.read_csv(dp)):,} entries)")

print(f"\nCoin: {COIN_NAME} | Working dir: {WORK_DIR}")
for label, f in [('Price', PRICE_CSV), ('Sentiment', SENT_FILE), ('News', NEWS_CSV)]:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  {label}: {os.path.basename(f)} ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING: {f}")

print(f"\nGPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
  Restored _finbert_cache.csv (42,867 entries)
  Restored _cryptobert_cache.csv (42,867 entries)
  Restored _news_finbert_cache.csv (228,446 entries)

Coin: ETH | Working dir: /content/eth_final
  Price: Ethereum_Price_Data.csv (0.3 MB)
  Sentiment: Ethereum_NLp_Data.csv (4.6 MB)
  News: coindesk_news_2020_2025.csv (229.3 MB)

GPU: True
  Tesla T4


In [ ]:
import pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')

def sma(s,n): return s.rolling(n,min_periods=n).mean()
def ema(s,n): return s.ewm(span=n,adjust=False,min_periods=n).mean()
def rsi(c,n=14):
    d=c.diff(); g=d.clip(lower=0).rolling(n,min_periods=n).mean()
    l=-d.clip(upper=0).rolling(n,min_periods=n).mean()
    return 100-(100/(1+g/(l+1e-10)))
def true_range(h,l,c):
    p=c.shift(1); return pd.concat([h-l,(h-p).abs(),(l-p).abs()],axis=1).max(axis=1)
def atr(h,l,c,n=14): return true_range(h,l,c).rolling(n,min_periods=n).mean()
def adx(h,l,c,n=14):
    up=h.diff(); dn=-l.diff()
    pdm=np.where((up>dn)&(up>0),up,0.0); ndm=np.where((dn>up)&(dn>0),dn,0.0)
    tr=true_range(h,l,c); an=tr.rolling(n,min_periods=n).mean()
    pdi=100*pd.Series(pdm,index=h.index).rolling(n,min_periods=n).mean()/(an+1e-10)
    ndi=100*pd.Series(ndm,index=h.index).rolling(n,min_periods=n).mean()/(an+1e-10)
    return (100*(pdi-ndi).abs()/(pdi+ndi+1e-10)).rolling(n,min_periods=n).mean()
def stoch(c,h,l,n=14,d=3):
    lo=l.rolling(n,min_periods=n).min(); hi=h.rolling(n,min_periods=n).max()
    k=100*(c-lo)/(hi-lo+1e-10); return k,k.rolling(d,min_periods=d).mean()
def mfi(h,l,c,v,n=14):
    tp=(h+l+c)/3; r=tp*v
    p=r.where(tp>tp.shift(1),0.0); neg=r.where(tp<tp.shift(1),0.0)
    ps=p.rolling(n,min_periods=n).sum(); ns=neg.rolling(n,min_periods=n).sum()
    return 100-(100/(1+ps/(ns+1e-10)))
def bollinger(c,n=20,k=2):
    m=c.rolling(n,min_periods=n).mean(); s=c.rolling(n,min_periods=n).std()
    u=m+k*s; lw=m-k*s; return (u-lw)/(m+1e-10),(c-lw)/(u-lw+1e-10)
def chaikin_mf(h,l,c,v,n=20):
    m=((c-l)-(h-c))/(h-l+1e-10)
    return (m*v).rolling(n,min_periods=n).sum()/(v.rolling(n,min_periods=n).sum()+1e-10)
def macd(c,f=12,s=26,sg=9):
    line=ema(c,f)-ema(c,s); sig=line.ewm(span=sg,adjust=False,min_periods=sg).mean()
    return line,sig,line-sig
def roc(c,n=10): return 100*(c-c.shift(n))/(c.shift(n)+1e-10)

df = pd.read_csv(PRICE_CSV, skiprows=[1])
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df.columns = ['date','close','high','low','open','volume']
for col in ['open','high','low','close','volume']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna().reset_index(drop=True)
print(f"Loaded ETH: {len(df)} rows ({df['date'].min().date()} to {df['date'].max().date()})")

c, h, l, o, v = df['close'], df['high'], df['low'], df['open'], df['volume']

df['sma_10_ratio'] = c / sma(c, 10) - 1
df['sma_20_ratio'] = c / sma(c, 20) - 1
df['sma_50_ratio'] = c / sma(c, 50) - 1
df['ema_12_ratio'] = c / ema(c, 12) - 1
df['ema_26_ratio'] = c / ema(c, 26) - 1
df['sma_cross_20_50'] = (sma(c,20) - sma(c,50)) / (sma(c,50) + 1e-10)
df['ema_cross_12_26'] = (ema(c,12) - ema(c,26)) / (ema(c,26) + 1e-10)

macd_line, macd_sig, macd_hist = macd(c)
df['macd_norm'] = macd_line / (c + 1e-10)
df['macd_signal_norm'] = macd_sig / (c + 1e-10)
df['macd_hist_norm'] = macd_hist / (c + 1e-10)

df['adx_14'] = adx(h, l, c, 14)
df['rsi_7'] = rsi(c, 7)
df['rsi_14'] = rsi(c, 14)
df['rsi_21'] = rsi(c, 21)
df['rsi_divergence'] = df['rsi_14'] - df['rsi_14'].rolling(5).mean()
df['stoch_k'], df['stoch_d'] = stoch(c, h, l)
df['stoch_kd_diff'] = df['stoch_k'] - df['stoch_d']

df['roc_3'] = roc(c, 3)
df['roc_7'] = roc(c, 7)
df['roc_14'] = roc(c, 14)
df['mfi_14'] = mfi(h, l, c, v, 14)

df['bb_width'], df['bb_pctb'] = bollinger(c)
df['atr_pct'] = atr(h, l, c, 14) / (c + 1e-10)
df['hist_vol_10'] = c.pct_change().rolling(10).std() * np.sqrt(365)
df['hist_vol_20'] = c.pct_change().rolling(20).std() * np.sqrt(365)
df['vol_of_vol'] = df['hist_vol_20'].rolling(20).std()

df['cmf_20'] = chaikin_mf(h, l, c, v, 20)
df['vol_sma_ratio'] = v / (v.rolling(20).mean() + 1e-10)
df['vol_zscore'] = (v - v.rolling(30).mean()) / (v.rolling(30).std() + 1e-10)
df['vol_acceleration'] = df['vol_sma_ratio'] - df['vol_sma_ratio'].rolling(5).mean()

df['log_ret_1'] = np.log(c / c.shift(1))
df['log_ret_3'] = np.log(c / c.shift(3))
df['log_ret_7'] = np.log(c / c.shift(7))
df['log_ret_14'] = np.log(c / c.shift(14))
df['ret_skew_14'] = df['log_ret_1'].rolling(14).skew()
df['ret_kurt_14'] = df['log_ret_1'].rolling(14).kurt()

df['hl_range'] = (h - l) / (c + 1e-10)
df['close_position'] = (c - l) / (h - l + 1e-10)
df['ret_zscore_20'] = (df['log_ret_1'] - df['log_ret_1'].rolling(20).mean()) / (df['log_ret_1'].rolling(20).std() + 1e-10)

print("\nBINARY TARGETS")
for h_days in [1, 7, 30]:
    fr = (df['close'].shift(-h_days) / df['close']) - 1
    df[f'target_{h_days}d'] = (fr > 0).astype(float)
    df[f'return_{h_days}d'] = fr
    df.loc[df.index[-h_days:], f'target_{h_days}d'] = np.nan
    df.loc[df.index[-h_days:], f'return_{h_days}d'] = np.nan
    t = df[f'target_{h_days}d'].dropna()
    print(f"  {h_days}d: {(t==1).sum()} UP, {(t==0).sum()} DOWN ({100*(t==1).mean():.1f}% UP)")

feat_cols = [col for col in df.columns
             if col not in ['date','open','high','low','close','volume']
             and not col.startswith('target_') and not col.startswith('return_')]
df_clean = df.dropna(subset=feat_cols).reset_index(drop=True)
df_clean.to_csv(f'outputs/{COIN_NAME.lower()}_features.csv', index=False)
print(f"\nSaved: {len(df_clean)} rows, {len(feat_cols)} features")
print(f"All features are RATIOS or NORMALIZED — no absolute price values that fail across regimes")

Loaded ETH: 2965 rows (2018-01-02 to 2026-02-13)

BINARY TARGETS
  1d: 1507 UP, 1457 DOWN (50.8% UP)
  7d: 1516 UP, 1442 DOWN (51.3% UP)
  30d: 1479 UP, 1456 DOWN (50.4% UP)

Saved: 2916 rows, 41 features
All features are RATIOS or NORMALIZED — no absolute price values that fail across regimes


In [ ]:
import requests, pandas as pd, numpy as np, time, os

OUT_CSV = f'outputs/onchain/{COIN_NAME.lower()}_onchain.csv'

def fetch_binance(symbol, limit=1000):
    url = 'https://api.binance.com/api/v3/klines'
    all_data = []
    end_time = int(time.time() * 1000)
    for attempt in range(10):
        params = {'symbol': symbol, 'interval': '1d', 'limit': limit, 'endTime': end_time}
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            if not data: break
            all_data = data + all_data
            end_time = data[0][0] - 1
            print(f"  Batch {attempt+1}: {len(data)} candles")
            if len(data) < limit: break
            time.sleep(0.3)
        except Exception as e:
            print(f"  Error: {e}"); break
    return all_data

print(f"Fetching {BINANCE_SYMBOL}...")
raw = fetch_binance(BINANCE_SYMBOL)

if raw and len(raw) > 100:
    df = pd.DataFrame(raw, columns=['open_time','open','high','low','close','volume',
        'close_time','quote_volume','n_trades','taker_buy_base','taker_buy_quote','ignore'])
    df['date'] = pd.to_datetime(df['open_time'], unit='ms').dt.normalize()
    for col in ['open','high','low','close','volume','quote_volume','n_trades','taker_buy_base','taker_buy_quote']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.drop_duplicates('date').sort_values('date').reset_index(drop=True)
    df = df[['date','close','volume','quote_volume','n_trades','taker_buy_base']].copy()
    df = df.rename(columns={'close':'price', 'quote_volume':'volume_usd'})
    df['market_cap'] = df['price'] * COIN_SUPPLY

    df['vol_to_mcap'] = df['volume_usd'] / (df['market_cap'] + 1e-10)
    df['vol_to_mcap_ma7'] = df['vol_to_mcap'].rolling(7, min_periods=1).mean()
    df['vol_to_mcap_change'] = df['vol_to_mcap'].pct_change().clip(-3, 3)

    df['mcap_log'] = np.log1p(df['market_cap'])
    df['mcap_zscore_30'] = (df['mcap_log'] - df['mcap_log'].rolling(30, min_periods=5).mean()) / (df['mcap_log'].rolling(30, min_periods=5).std() + 1e-10)

    df['volume_surge'] = df['volume_usd'] / (df['volume_usd'].rolling(30, min_periods=5).mean() + 1e-10)
    df['volume_zscore_30'] = (df['volume_usd'] - df['volume_usd'].rolling(30, min_periods=5).mean()) / (df['volume_usd'].rolling(30, min_periods=5).std() + 1e-10)

    df['price_change_7d'] = df['price'].pct_change(7)
    df['volume_change_7d'] = df['volume_usd'].pct_change(7).clip(-3, 3)
    df['price_vol_divergence'] = df['price_change_7d'] - df['volume_change_7d']

    df['taker_buy_ratio'] = df['taker_buy_base'] / (df['volume'] + 1e-10)
    df['taker_buy_ratio_ma7'] = df['taker_buy_ratio'].rolling(7, min_periods=1).mean()
    df['trades_per_day_zscore'] = (df['n_trades'] - df['n_trades'].rolling(30, min_periods=5).mean()) / (df['n_trades'].rolling(30, min_periods=5).std() + 1e-10)

    onchain_cols = ['vol_to_mcap','vol_to_mcap_ma7','vol_to_mcap_change','mcap_zscore_30',
                    'volume_surge','volume_zscore_30','price_vol_divergence',
                    'taker_buy_ratio','taker_buy_ratio_ma7','trades_per_day_zscore']
    for col in onchain_cols:
        df[col] = df[col].fillna(0).replace([np.inf, -np.inf], 0)
    df.to_csv(OUT_CSV, index=False)
    print(f"\nSaved {len(df)} rows ({len(onchain_cols)} on-chain features)")
else:
    print("Binance failed, using fallback")
    pdf = pd.read_csv(PRICE_CSV, skiprows=[1])
    pdf['Date'] = pd.to_datetime(pdf['Date'])
    pdf = pdf.sort_values('Date').reset_index(drop=True)
    pdf.columns = ['date','close','high','low','open','volume']
    for col in ['open','high','low','close','volume']:
        pdf[col] = pd.to_numeric(pdf[col], errors='coerce')
    pdf = pdf.dropna().reset_index(drop=True)
    df = pd.DataFrame({'date': pdf['date'].values, 'price': pdf['close'].values,
                        'volume_usd': pdf['volume'].values * pdf['close'].values})
    df['market_cap'] = df['price'] * COIN_SUPPLY
    df['vol_to_mcap'] = df['volume_usd'] / (df['market_cap'] + 1e-10)
    df['vol_to_mcap_ma7'] = df['vol_to_mcap'].rolling(7, min_periods=1).mean()
    df['vol_to_mcap_change'] = df['vol_to_mcap'].pct_change().clip(-3, 3)
    df['mcap_log'] = np.log1p(df['market_cap'])
    df['mcap_zscore_30'] = (df['mcap_log'] - df['mcap_log'].rolling(30, min_periods=5).mean()) / (df['mcap_log'].rolling(30, min_periods=5).std() + 1e-10)
    df['volume_surge'] = df['volume_usd'] / (df['volume_usd'].rolling(30, min_periods=5).mean() + 1e-10)
    df['volume_zscore_30'] = (df['volume_usd'] - df['volume_usd'].rolling(30, min_periods=5).mean()) / (df['volume_usd'].rolling(30, min_periods=5).std() + 1e-10)
    df['price_change_7d'] = df['price'].pct_change(7)
    df['volume_change_7d'] = df['volume_usd'].pct_change(7).clip(-3, 3)
    df['price_vol_divergence'] = df['price_change_7d'] - df['volume_change_7d']
    df['taker_buy_ratio'] = 0.5
    df['taker_buy_ratio_ma7'] = 0.5
    df['trades_per_day_zscore'] = 0.0
    df = df.fillna(0)
    df.to_csv(OUT_CSV, index=False)
    print(f"Fallback saved {len(df)} rows")

Fetching ETHUSDT...
  Batch 1: 1000 candles
  Batch 2: 1000 candles
  Batch 3: 1000 candles
  Batch 4: 179 candles

Saved 3179 rows (10 on-chain features)


In [ ]:
import pandas as pd, numpy as np

features = pd.read_csv(f'outputs/{COIN_NAME.lower()}_features.csv', parse_dates=['date'])
onchain = pd.read_csv(f'outputs/onchain/{COIN_NAME.lower()}_onchain.csv', parse_dates=['date'])

ONCHAIN_FEATURES = ['vol_to_mcap','vol_to_mcap_ma7','vol_to_mcap_change','mcap_zscore_30',
                    'volume_surge','volume_zscore_30','price_vol_divergence',
                    'taker_buy_ratio','taker_buy_ratio_ma7','trades_per_day_zscore']

merged = features.merge(onchain[['date'] + ONCHAIN_FEATURES], on='date', how='left')
for col in ONCHAIN_FEATURES:
    merged[col] = merged[col].ffill().fillna(0)

merged.to_csv(f'outputs/{COIN_NAME.lower()}_features.csv', index=False)
print(f"Merged: {len(merged)} rows × {len(merged.columns)} columns")

Merged: 2916 rows × 63 columns


In [ ]:
import pandas as pd, numpy as np, re, os, torch, shutil

FINBERT_CACHE = 'outputs/_finbert_cache.csv'
CRYPTOBERT_CACHE = 'outputs/_cryptobert_cache.csv'

sdf = pd.read_csv(SENT_FILE)
sdf = sdf[sdf['lang']=='en'].reset_index(drop=True)
sdf['created_at'] = pd.to_datetime(sdf['created_at'], errors='coerce', utc=True)
sdf = sdf.dropna(subset=['created_at']).reset_index(drop=True)
sdf['date'] = sdf['created_at'].dt.tz_convert(None).dt.normalize()
sdf['text'] = sdf['text'].astype(str)
sdf = sdf[sdf['text'].str.strip().str.len() > 10].reset_index(drop=True)

def clean(t):
    t = re.sub(r'http\S+|www\.\S+', '', t)
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#(\w+)', r'\1', t)
    return re.sub(r'\s+', ' ', t).strip()

sdf['text_clean'] = sdf['text'].apply(clean)
sdf = sdf[sdf['text_clean'].str.len() > 5].reset_index(drop=True)
sdf['coin'] = COIN_NAME
sdf['tweet_id_coin'] = COIN_NAME + '_' + sdf['tweet_id'].astype(str)
print(f"Tweets: {len(sdf):,}")

def run_model(model_name, cache_path, prefix):
    id_col = 'tweet_id_coin'
    if os.path.exists(cache_path):
        cache = pd.read_csv(cache_path)
        if f'{prefix}_pos' in cache.columns and id_col in cache.columns:
            sdf_m = sdf.merge(cache[[id_col, f'{prefix}_pos', f'{prefix}_neu', f'{prefix}_neg']], on=id_col, how='left')
        else:
            sdf_m = sdf.copy()
            for c in [f'{prefix}_pos', f'{prefix}_neu', f'{prefix}_neg']: sdf_m[c] = np.nan
    else:
        sdf_m = sdf.copy()
        for c in [f'{prefix}_pos', f'{prefix}_neu', f'{prefix}_neg']: sdf_m[c] = np.nan

    pending = sdf_m[sdf_m[f'{prefix}_pos'].isna()].reset_index(drop=True)
    print(f"  {prefix}: {len(pending):,} to score")

    if len(pending) > 0:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        tok = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()
        id2label = {k:v.lower() for k,v in model.config.id2label.items()}
        def map_label(lbl):
            l = lbl.lower()
            if 'pos' in l or 'bull' in l or l=='label_2': return 'pos'
            if 'neg' in l or 'bear' in l or l=='label_0': return 'neg'
            return 'neu'
        BATCH = 64 if torch.cuda.is_available() else 16
        results = []
        with torch.no_grad():
            for start in range(0, len(pending), BATCH):
                bt = pending['text_clean'].iloc[start:start+BATCH].tolist()
                bi = pending[id_col].iloc[start:start+BATCH].tolist()
                enc = tok(bt, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
                probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
                for i, tid in enumerate(bi):
                    row = {id_col: tid, f'{prefix}_pos':0.0, f'{prefix}_neu':0.0, f'{prefix}_neg':0.0}
                    for j, lbl in id2label.items():
                        row[f'{prefix}_{map_label(lbl)}'] += float(probs[i, j])
                    results.append(row)
                if (start // BATCH) % 10 == 0:
                    print(f"    {start+len(bt):,}/{len(pending):,}")
                if len(results) >= 500:
                    pp = pd.DataFrame(results)
                    if os.path.exists(cache_path):
                        pp = pd.concat([pd.read_csv(cache_path), pp]).drop_duplicates(id_col)
                    pp.to_csv(cache_path, index=False)
                    results = []
        if results:
            pp = pd.DataFrame(results)
            if os.path.exists(cache_path):
                pp = pd.concat([pd.read_csv(cache_path), pp]).drop_duplicates(id_col)
            pp.to_csv(cache_path, index=False)
        del model, tok
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        sdf_m = sdf_m.drop(columns=[c for c in sdf_m.columns if c.startswith(prefix+'_')], errors='ignore')
        sdf_m = sdf_m.merge(pd.read_csv(cache_path), on=id_col, how='left')
    return sdf_m[[id_col, f'{prefix}_pos', f'{prefix}_neu', f'{prefix}_neg']]

fb = run_model('ProsusAI/finbert', FINBERT_CACHE, 'fb')
cb = run_model('ElKulako/cryptobert', CRYPTOBERT_CACHE, 'cb')

for c in [FINBERT_CACHE, CRYPTOBERT_CACHE]:
    if os.path.exists(c): shutil.copy(c, f"{DRIVE_DIR}/{os.path.basename(c)}")

sdf = sdf.merge(fb, on='tweet_id_coin', how='left').merge(cb, on='tweet_id_coin', how='left')
sdf['pos'] = (sdf['fb_pos'] + sdf['cb_pos']) / 2
sdf['neu'] = (sdf['fb_neu'] + sdf['cb_neu']) / 2
sdf['neg'] = (sdf['fb_neg'] + sdf['cb_neg']) / 2
sdf['compound'] = sdf['pos'] - sdf['neg']
sdf['engagement'] = np.log1p(sdf['retweets'].fillna(0) + sdf['favorites'].fillna(0)*0.5
                              + sdf['replies'].fillna(0) + sdf['quotes'].fillna(0)*2)
sdf['weight'] = sdf['engagement'] * np.where(sdf['user_info.verified'].fillna(False), 1.2, 1.0)
sdf.loc[sdf['weight']==0, 'weight'] = 1.0

print(f"\nClass balance check on tweets:")
print(f"  Positive: {(sdf['compound'] > 0.1).sum()} ({100*(sdf['compound']>0.1).mean():.1f}%)")
print(f"  Negative: {(sdf['compound'] < -0.1).sum()} ({100*(sdf['compound']<-0.1).mean():.1f}%)")
print(f"  Neutral:  {((sdf['compound']>=-0.1) & (sdf['compound']<=0.1)).sum()}")

def wagg(g):
    w = g['weight'].values; ws = w.sum()
    if ws == 0:
        return pd.Series({'sent_pos':g['pos'].mean(),'sent_neu':g['neu'].mean(),
                          'sent_neg':g['neg'].mean(),'sent_compound':g['compound'].mean(),
                          'tweet_count':len(g),'engagement_sum':0.0,
                          'sent_pos_share': (g['compound']>0.1).mean(),
                          'sent_neg_share': (g['compound']<-0.1).mean()})
    return pd.Series({'sent_pos':(g['pos']*w).sum()/ws,'sent_neu':(g['neu']*w).sum()/ws,
                      'sent_neg':(g['neg']*w).sum()/ws,'sent_compound':(g['compound']*w).sum()/ws,
                      'tweet_count':len(g),'engagement_sum':float(ws),
                      'sent_pos_share': (g['compound']>0.1).mean(),
                      'sent_neg_share': (g['compound']<-0.1).mean()})

daily = sdf.groupby('date').apply(wagg).reset_index()
price_df = pd.read_csv(f'outputs/{COIN_NAME.lower()}_features.csv', parse_dates=['date'])
all_dates = pd.DataFrame({'date': price_df['date'].sort_values().unique()})
merged = all_dates.merge(daily, on='date', how='left').sort_values('date').reset_index(drop=True)

for col in ['sent_pos','sent_neu','sent_neg','sent_compound','sent_pos_share','sent_neg_share']:
    vals = merged[col].values.copy()
    last, streak = np.nan, 0
    for i in range(len(vals)):
        if np.isnan(vals[i]):
            if not np.isnan(last):
                streak += 1; decay = 0.7**streak
                vals[i] = last*decay if 'compound' in col or 'share' in col else last*decay+(1-decay)*(1/3)
        else: last, streak = vals[i], 0
    merged[col] = vals

merged['tweet_count'] = merged['tweet_count'].fillna(0)
merged['engagement_sum'] = merged['engagement_sum'].fillna(0)
merged['sent_pos'] = merged['sent_pos'].fillna(1/3)
merged['sent_neu'] = merged['sent_neu'].fillna(1/3)
merged['sent_neg'] = merged['sent_neg'].fillna(1/3)
merged['sent_compound'] = merged['sent_compound'].fillna(0.0)
merged['sent_pos_share'] = merged['sent_pos_share'].fillna(0.5)
merged['sent_neg_share'] = merged['sent_neg_share'].fillna(0.5)
merged['sent_polarity'] = merged['sent_pos_share'] - merged['sent_neg_share']
merged['sent_ma_3'] = merged['sent_compound'].rolling(3, min_periods=1).mean()
merged['sent_ma_7'] = merged['sent_compound'].rolling(7, min_periods=1).mean()
merged['sent_std_7'] = merged['sent_compound'].rolling(7, min_periods=1).std().fillna(0)
merged['sent_momentum'] = merged['sent_compound'] - merged['sent_ma_7']
m = merged['sent_compound'].rolling(30, min_periods=5).mean()
s = merged['sent_compound'].rolling(30, min_periods=5).std()
merged['sent_zscore'] = ((merged['sent_compound']-m)/(s+1e-6)).fillna(0)
merged['tweet_count_ratio'] = merged['tweet_count']/(merged['tweet_count'].rolling(7, min_periods=1).mean()+1e-6)

merged.to_csv(f'outputs/{COIN_NAME.lower()}_sentiment_daily.csv', index=False)
print(f"Saved sentiment: {len(merged)} rows")

Tweets: 12,389
  fb: 0 to score
  cb: 0 to score

Class balance check on tweets:
  Positive: 10906 (88.0%)
  Negative: 644 (5.2%)
  Neutral:  839
Saved sentiment: 2916 rows


In [ ]:
import pandas as pd, numpy as np, re, os, torch, shutil

FINBERT_CACHE = 'outputs/_finbert_cache.csv'
CRYPTOBERT_CACHE = 'outputs/_cryptobert_cache.csv'

sdf = pd.read_csv(SENT_FILE)
sdf = sdf[sdf['lang']=='en'].reset_index(drop=True)
sdf['created_at'] = pd.to_datetime(sdf['created_at'], errors='coerce', utc=True)
sdf = sdf.dropna(subset=['created_at']).reset_index(drop=True)
sdf['date'] = sdf['created_at'].dt.tz_convert(None).dt.normalize()
sdf['text'] = sdf['text'].astype(str)
sdf = sdf[sdf['text'].str.strip().str.len() > 10].reset_index(drop=True)

def clean(t):
    t = re.sub(r'http\S+|www\.\S+', '', t)
    t = re.sub(r'@\w+', '', t)
    t = re.sub(r'#(\w+)', r'\1', t)
    return re.sub(r'\s+', ' ', t).strip()

sdf['text_clean'] = sdf['text'].apply(clean)
sdf = sdf[sdf['text_clean'].str.len() > 5].reset_index(drop=True)
sdf['coin'] = COIN_NAME
sdf['tweet_id_coin'] = COIN_NAME + '_' + sdf['tweet_id'].astype(str)
print(f"Tweets: {len(sdf):,}")

def run_model(model_name, cache_path, prefix):
    id_col = 'tweet_id_coin'
    if os.path.exists(cache_path):
        cache = pd.read_csv(cache_path)
        if f'{prefix}_pos' in cache.columns and id_col in cache.columns:
            sdf_m = sdf.merge(cache[[id_col, f'{prefix}_pos', f'{prefix}_neu', f'{prefix}_neg']], on=id_col, how='left')
        else:
            sdf_m = sdf.copy()
            for c in [f'{prefix}_pos', f'{prefix}_neu', f'{prefix}_neg']: sdf_m[c] = np.nan
    else:
        sdf_m = sdf.copy()
        for c in [f'{prefix}_pos', f'{prefix}_neu', f'{prefix}_neg']: sdf_m[c] = np.nan

    pending = sdf_m[sdf_m[f'{prefix}_pos'].isna()].reset_index(drop=True)
    print(f"  {prefix}: {len(pending):,} to score")

    if len(pending) > 0:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        tok = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device).eval()
        id2label = {k:v.lower() for k,v in model.config.id2label.items()}
        def map_label(lbl):
            l = lbl.lower()
            if 'pos' in l or 'bull' in l or l=='label_2': return 'pos'
            if 'neg' in l or 'bear' in l or l=='label_0': return 'neg'
            return 'neu'
        BATCH = 64 if torch.cuda.is_available() else 16
        results = []
        with torch.no_grad():
            for start in range(0, len(pending), BATCH):
                bt = pending['text_clean'].iloc[start:start+BATCH].tolist()
                bi = pending[id_col].iloc[start:start+BATCH].tolist()
                enc = tok(bt, padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
                probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
                for i, tid in enumerate(bi):
                    row = {id_col: tid, f'{prefix}_pos':0.0, f'{prefix}_neu':0.0, f'{prefix}_neg':0.0}
                    for j, lbl in id2label.items():
                        row[f'{prefix}_{map_label(lbl)}'] += float(probs[i, j])
                    results.append(row)
                if (start // BATCH) % 10 == 0:
                    print(f"    {start+len(bt):,}/{len(pending):,}")
                if len(results) >= 500:
                    pp = pd.DataFrame(results)
                    if os.path.exists(cache_path):
                        pp = pd.concat([pd.read_csv(cache_path), pp]).drop_duplicates(id_col)
                    pp.to_csv(cache_path, index=False)
                    results = []
        if results:
            pp = pd.DataFrame(results)
            if os.path.exists(cache_path):
                pp = pd.concat([pd.read_csv(cache_path), pp]).drop_duplicates(id_col)
            pp.to_csv(cache_path, index=False)
        del model, tok
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        sdf_m = sdf_m.drop(columns=[c for c in sdf_m.columns if c.startswith(prefix+'_')], errors='ignore')
        sdf_m = sdf_m.merge(pd.read_csv(cache_path), on=id_col, how='left')
    return sdf_m[[id_col, f'{prefix}_pos', f'{prefix}_neu', f'{prefix}_neg']]

fb = run_model('ProsusAI/finbert', FINBERT_CACHE, 'fb')
cb = run_model('ElKulako/cryptobert', CRYPTOBERT_CACHE, 'cb')

for c in [FINBERT_CACHE, CRYPTOBERT_CACHE]:
    if os.path.exists(c): shutil.copy(c, f"{DRIVE_DIR}/{os.path.basename(c)}")

sdf = sdf.merge(fb, on='tweet_id_coin', how='left').merge(cb, on='tweet_id_coin', how='left')
sdf['pos'] = (sdf['fb_pos'] + sdf['cb_pos']) / 2
sdf['neu'] = (sdf['fb_neu'] + sdf['cb_neu']) / 2
sdf['neg'] = (sdf['fb_neg'] + sdf['cb_neg']) / 2
sdf['compound'] = sdf['pos'] - sdf['neg']

print(f"\nBEFORE BALANCING:")
print(f"  Positive (>0.1): {(sdf['compound'] > 0.1).sum()} ({100*(sdf['compound']>0.1).mean():.1f}%)")
print(f"  Negative (<-0.1): {(sdf['compound'] < -0.1).sum()} ({100*(sdf['compound']<-0.1).mean():.1f}%)")
print(f"  Neutral: {((sdf['compound']>=-0.1) & (sdf['compound']<=0.1)).sum()}")

print(f"\nFix 1: Removing FinBERT positive bias via threshold recalibration")
threshold_pos = sdf['compound'].quantile(0.66)
threshold_neg = sdf['compound'].quantile(0.33)
print(f"  Pos threshold (66th percentile): {threshold_pos:.3f}")
print(f"  Neg threshold (33rd percentile): {threshold_neg:.3f}")

sdf['compound_balanced'] = sdf['compound'].copy()
sdf.loc[sdf['compound'] > threshold_pos, 'sentiment_class'] = 'pos'
sdf.loc[sdf['compound'] < threshold_neg, 'sentiment_class'] = 'neg'
sdf.loc[(sdf['compound'] >= threshold_neg) & (sdf['compound'] <= threshold_pos), 'sentiment_class'] = 'neu'

print(f"\nAFTER PERCENTILE-BASED REBALANCING:")
print(f"  Positive: {(sdf['sentiment_class']=='pos').sum()} ({100*(sdf['sentiment_class']=='pos').mean():.1f}%)")
print(f"  Negative: {(sdf['sentiment_class']=='neg').sum()} ({100*(sdf['sentiment_class']=='neg').mean():.1f}%)")
print(f"  Neutral: {(sdf['sentiment_class']=='neu').sum()} ({100*(sdf['sentiment_class']=='neu').mean():.1f}%)")

print(f"\nFix 2: Engagement-weighted balancing")
sdf['engagement'] = np.log1p(sdf['retweets'].fillna(0) + sdf['favorites'].fillna(0)*0.5
                              + sdf['replies'].fillna(0) + sdf['quotes'].fillna(0)*2)
sdf['weight'] = sdf['engagement'] * np.where(sdf['user_info.verified'].fillna(False), 1.2, 1.0)
sdf.loc[sdf['weight']==0, 'weight'] = 1.0

sdf['balance_weight'] = 1.0
neg_count = (sdf['sentiment_class']=='neg').sum()
pos_count = (sdf['sentiment_class']=='pos').sum()
neu_count = (sdf['sentiment_class']=='neu').sum()
total = len(sdf)
sdf.loc[sdf['sentiment_class']=='neg', 'balance_weight'] = total / (3 * max(neg_count, 1))
sdf.loc[sdf['sentiment_class']=='pos', 'balance_weight'] = total / (3 * max(pos_count, 1))
sdf.loc[sdf['sentiment_class']=='neu', 'balance_weight'] = total / (3 * max(neu_count, 1))
sdf['final_weight'] = sdf['weight'] * sdf['balance_weight']

print(f"  Balance weights: pos={total/(3*max(pos_count,1)):.3f} neg={total/(3*max(neg_count,1)):.3f} neu={total/(3*max(neu_count,1)):.3f}")

print(f"\nFix 3: Computing balanced daily aggregations")

def wagg(g):
    w = g['final_weight'].values
    ws = w.sum()
    if ws == 0:
        w = np.ones_like(w); ws = len(w)

    sent_pos_balanced = (g['pos'] * w).sum() / ws
    sent_neu_balanced = (g['neu'] * w).sum() / ws
    sent_neg_balanced = (g['neg'] * w).sum() / ws
    sent_compound_balanced = sent_pos_balanced - sent_neg_balanced

    pos_count = (g['sentiment_class']=='pos').sum()
    neg_count = (g['sentiment_class']=='neg').sum()
    neu_count = (g['sentiment_class']=='neu').sum()
    total = len(g)

    return pd.Series({
        'sent_pos': sent_pos_balanced,
        'sent_neu': sent_neu_balanced,
        'sent_neg': sent_neg_balanced,
        'sent_compound': sent_compound_balanced,
        'sent_pos_share': pos_count / max(total, 1),
        'sent_neg_share': neg_count / max(total, 1),
        'sent_neu_share': neu_count / max(total, 1),
        'sent_polarity_ratio': (pos_count - neg_count) / max(pos_count + neg_count, 1),
        'tweet_count': total,
        'engagement_sum': float(g['weight'].sum()),
        'sent_disagreement': g['compound'].std() if len(g) > 1 else 0,
        'sent_extreme_pos': (g['pos'] > 0.7).sum() / max(total, 1),
        'sent_extreme_neg': (g['neg'] > 0.5).sum() / max(total, 1),
    })

daily = sdf.groupby('date').apply(wagg).reset_index()
price_df = pd.read_csv(f'outputs/{COIN_NAME.lower()}_features.csv', parse_dates=['date'])
all_dates = pd.DataFrame({'date': price_df['date'].sort_values().unique()})
merged = all_dates.merge(daily, on='date', how='left').sort_values('date').reset_index(drop=True)

print(f"\nDaily sentiment statistics (after balancing):")
print(f"  Mean compound: {merged['sent_compound'].mean():.4f}")
print(f"  Std compound:  {merged['sent_compound'].std():.4f}")
print(f"  Range:         [{merged['sent_compound'].min():.3f}, {merged['sent_compound'].max():.3f}]")
print(f"  Mean pos_share: {merged['sent_pos_share'].mean():.3f}")
print(f"  Mean neg_share: {merged['sent_neg_share'].mean():.3f}")

for col in ['sent_pos','sent_neu','sent_neg','sent_compound','sent_pos_share','sent_neg_share','sent_neu_share',
            'sent_polarity_ratio','sent_disagreement','sent_extreme_pos','sent_extreme_neg']:
    vals = merged[col].values.copy()
    last, streak = np.nan, 0
    for i in range(len(vals)):
        if np.isnan(vals[i]):
            if not np.isnan(last):
                streak += 1
                decay = 0.7**streak
                if 'compound' in col or 'polarity' in col or 'disagreement' in col:
                    vals[i] = last * decay
                else:
                    vals[i] = last * decay + (1-decay) * 0.33
        else:
            last, streak = vals[i], 0
    merged[col] = vals

merged['tweet_count'] = merged['tweet_count'].fillna(0)
merged['engagement_sum'] = merged['engagement_sum'].fillna(0)
fill_defaults = {
    'sent_pos': 1/3, 'sent_neu': 1/3, 'sent_neg': 1/3,
    'sent_compound': 0.0, 'sent_pos_share': 0.33, 'sent_neg_share': 0.33,
    'sent_neu_share': 0.33, 'sent_polarity_ratio': 0.0,
    'sent_disagreement': 0.0, 'sent_extreme_pos': 0.0, 'sent_extreme_neg': 0.0
}
for col, default in fill_defaults.items():
    merged[col] = merged[col].fillna(default)

merged['sent_ma_3'] = merged['sent_compound'].rolling(3, min_periods=1).mean()
merged['sent_ma_7'] = merged['sent_compound'].rolling(7, min_periods=1).mean()
merged['sent_std_7'] = merged['sent_compound'].rolling(7, min_periods=1).std().fillna(0)
merged['sent_momentum'] = merged['sent_compound'] - merged['sent_ma_7']
m = merged['sent_compound'].rolling(30, min_periods=5).mean()
s = merged['sent_compound'].rolling(30, min_periods=5).std()
merged['sent_zscore'] = ((merged['sent_compound']-m)/(s+1e-6)).fillna(0)
merged['tweet_count_ratio'] = merged['tweet_count']/(merged['tweet_count'].rolling(7, min_periods=1).mean()+1e-6)
merged['polarity_momentum'] = merged['sent_polarity_ratio'] - merged['sent_polarity_ratio'].rolling(7, min_periods=1).mean()

merged.to_csv(f'outputs/{COIN_NAME.lower()}_sentiment_daily.csv', index=False)

print(f"\nFinal feature statistics (variance check — high std = good signal):")
for col in ['sent_compound', 'sent_polarity_ratio', 'sent_pos_share', 'sent_neg_share',
            'sent_disagreement', 'sent_extreme_pos', 'sent_zscore', 'polarity_momentum']:
    print(f"  {col:25s}: std={merged[col].std():.4f}  range=[{merged[col].min():+.3f}, {merged[col].max():+.3f}]")

print(f"\nSaved: {len(merged)} rows with balanced sentiment features")

Tweets: 12,389
  fb: 0 to score
  cb: 0 to score

BEFORE BALANCING:
  Positive (>0.1): 10906 (88.0%)
  Negative (<-0.1): 644 (5.2%)
  Neutral: 839

Fix 1: Removing FinBERT positive bias via threshold recalibration
  Pos threshold (66th percentile): 0.384
  Neg threshold (33rd percentile): 0.252

AFTER PERCENTILE-BASED REBALANCING:
  Positive: 4212 (34.0%)
  Negative: 4089 (33.0%)
  Neutral: 4088 (33.0%)

Fix 2: Engagement-weighted balancing
  Balance weights: pos=0.980 neg=1.010 neu=1.010

Fix 3: Computing balanced daily aggregations

Daily sentiment statistics (after balancing):
  Mean compound: 0.2979
  Std compound:  0.1536
  Range:         [-0.955, 0.831]
  Mean pos_share: 0.333
  Mean neg_share: 0.336

Final feature statistics (variance check — high std = good signal):
  sent_compound            : std=0.1550  range=[-0.955, +0.831]
  sent_polarity_ratio      : std=0.6592  range=[-1.000, +1.000]
  sent_pos_share           : std=0.2957  range=[+0.000, +1.000]
  sent_neg_share       

In [ ]:
import pandas as pd, numpy as np, os, re, hashlib, torch, shutil, time

NEWS_FEATURES_OUT = 'outputs/news/news_daily_features.csv'
NEWS_CACHE = 'outputs/news/_news_finbert_cache.csv'
DRIVE_NEWS_CACHE = f'{DRIVE_DIR}/_news_finbert_cache.csv'

if os.path.exists(DRIVE_NEWS_CACHE):
    shutil.copy(DRIVE_NEWS_CACHE, NEWS_CACHE)
    print(f"Restored news cache: {len(pd.read_csv(NEWS_CACHE)):,} entries")

print("Loading CoinDesk news...")
news_df = pd.read_csv(NEWS_CSV)
print(f"  Total: {len(news_df):,}")

news_df['date'] = pd.to_datetime(news_df['published_on'], errors='coerce').dt.normalize()
news_df = news_df.dropna(subset=['date']).reset_index(drop=True)
news_df = news_df[(news_df['date'] >= '2019-01-01') & (news_df['date'] <= '2025-12-31')].reset_index(drop=True)

target_tag = 'ETH'
news_df['tags'] = news_df['tags'].fillna('')
news_df['categories'] = news_df['categories'].fillna('')
news_df['title'] = news_df['title'].fillna('')
news_df['body'] = news_df['body'].fillna('')

def is_relevant(row):
    text = f"{row['tags']} {row['categories']} {row['title']}".upper()
    return target_tag in text

news_df = news_df[news_df.apply(is_relevant, axis=1)].reset_index(drop=True)
print(f"  After ETH filter: {len(news_df):,}")

spam_keywords = ['presale', 'sponsored', 'disclaimer', 'lightchain', 'jump into',
                 "don't miss out", 'token presale', 'whitepaper']
def is_spam(row):
    text = f"{row['title']} {row['body'][:500]}".lower()
    return sum(kw in text for kw in spam_keywords) >= 2

news_df = news_df[~news_df.apply(is_spam, axis=1)].reset_index(drop=True)
print(f"  After spam filter: {len(news_df):,}")

def clean_html(text):
    text = re.sub(r'&apos;', "'", str(text))
    text = re.sub(r'&quot;', '"', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

news_df['title_clean'] = news_df['title'].apply(clean_html)
news_df['body_clean'] = news_df['body'].apply(lambda x: clean_html(x)[:500])
news_df['combined_text'] = news_df['title_clean'] + ". " + news_df['body_clean']
news_df = news_df[news_df['combined_text'].str.len() > 30].reset_index(drop=True)

def make_id(row):
    raw = f"{row['date']}_{row['source']}_{str(row['title_clean'])[:100]}"
    return hashlib.md5(raw.encode()).hexdigest()[:16]

news_df['article_id'] = news_df.apply(make_id, axis=1)
news_df = news_df.drop_duplicates('article_id').reset_index(drop=True)

if os.path.exists(NEWS_CACHE):
    cache = pd.read_csv(NEWS_CACHE)
    if 'article_id' in cache.columns and 'news_pos' in cache.columns:
        news_df = news_df.merge(cache[['article_id','news_pos','news_neu','news_neg']], on='article_id', how='left')
    else:
        for c in ['news_pos','news_neu','news_neg']: news_df[c] = np.nan
else:
    for c in ['news_pos','news_neu','news_neg']: news_df[c] = np.nan

pending = news_df[news_df['news_pos'].isna()].reset_index(drop=True)
print(f"Total: {len(news_df):,} | Cached: {len(news_df)-len(pending):,} | Pending: {len(pending):,}")

if len(pending) > 0:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    print("Scoring with FinBERT...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert').to(device).eval()
    BATCH = 64 if torch.cuda.is_available() else 16
    results = []
    t_start = time.time()
    with torch.no_grad():
        for start in range(0, len(pending), BATCH):
            batch_text = pending['combined_text'].iloc[start:start+BATCH].tolist()
            batch_ids = pending['article_id'].iloc[start:start+BATCH].tolist()
            try:
                enc = tok(batch_text, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
                probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
                for i, aid in enumerate(batch_ids):
                    results.append({'article_id': aid, 'news_pos': float(probs[i,0]),
                                    'news_neg': float(probs[i,1]), 'news_neu': float(probs[i,2])})
            except: continue
            if (start // BATCH) % 50 == 0:
                elapsed = time.time() - t_start
                rate = (start + len(batch_text)) / max(elapsed, 1)
                eta = (len(pending) - start - len(batch_text)) / max(rate, 1) / 60
                print(f"   {start+len(batch_text):,}/{len(pending):,} | {rate:.0f}/s | ETA: {eta:.1f} min")
            if len(results) >= 2000:
                pp = pd.DataFrame(results)
                if os.path.exists(NEWS_CACHE):
                    pp = pd.concat([pd.read_csv(NEWS_CACHE), pp]).drop_duplicates('article_id')
                pp.to_csv(NEWS_CACHE, index=False)
                shutil.copy(NEWS_CACHE, DRIVE_NEWS_CACHE)
                results = []
    if results:
        pp = pd.DataFrame(results)
        if os.path.exists(NEWS_CACHE):
            pp = pd.concat([pd.read_csv(NEWS_CACHE), pp]).drop_duplicates('article_id')
        pp.to_csv(NEWS_CACHE, index=False)
    shutil.copy(NEWS_CACHE, DRIVE_NEWS_CACHE)
    del model, tok
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    news_df = news_df.drop(columns=['news_pos','news_neu','news_neg'], errors='ignore')
    news_df = news_df.merge(pd.read_csv(NEWS_CACHE), on='article_id', how='left')

news_df['news_compound'] = news_df['news_pos'] - news_df['news_neg']
news_df['is_pos'] = (news_df['news_compound'] > 0.1).astype(int)
news_df['is_neg'] = (news_df['news_compound'] < -0.1).astype(int)

print(f"\nNews class balance:")
print(f"  Positive articles: {news_df['is_pos'].sum()} ({100*news_df['is_pos'].mean():.1f}%)")
print(f"  Negative articles: {news_df['is_neg'].sum()} ({100*news_df['is_neg'].mean():.1f}%)")

def agg_daily(g):
    return pd.Series({
        'news_count': len(g),
        'news_sent_pos': g['news_pos'].mean(),
        'news_sent_neu': g['news_neu'].mean(),
        'news_sent_neg': g['news_neg'].mean(),
        'news_sent_compound': g['news_compound'].mean(),
        'news_pos_share': g['is_pos'].mean(),
        'news_neg_share': g['is_neg'].mean(),
        'news_sent_std': g['news_compound'].std() if len(g) > 1 else 0
    })

daily_news = news_df.groupby('date').apply(agg_daily).reset_index()
price_df = pd.read_csv(f'outputs/{COIN_NAME.lower()}_features.csv', parse_dates=['date'])
all_dates = pd.DataFrame({'date': sorted(price_df['date'].unique())})
daily_news = all_dates.merge(daily_news, on='date', how='left')

for col in ['news_sent_pos','news_sent_neu','news_sent_neg','news_sent_compound','news_pos_share','news_neg_share']:
    vals = daily_news[col].values.copy()
    last, streak = np.nan, 0
    for i in range(len(vals)):
        if np.isnan(vals[i]):
            if not np.isnan(last):
                streak += 1
                decay = 0.7 ** streak
                vals[i] = last * decay if 'compound' in col or 'share' in col else last * decay + (1 - decay) * (1/3)
        else: last, streak = vals[i], 0
    daily_news[col] = vals

daily_news['news_count'] = daily_news['news_count'].fillna(0)
daily_news['news_sent_pos'] = daily_news['news_sent_pos'].fillna(1/3)
daily_news['news_sent_neu'] = daily_news['news_sent_neu'].fillna(1/3)
daily_news['news_sent_neg'] = daily_news['news_sent_neg'].fillna(1/3)
daily_news['news_sent_compound'] = daily_news['news_sent_compound'].fillna(0.0)
daily_news['news_pos_share'] = daily_news['news_pos_share'].fillna(0.5)
daily_news['news_neg_share'] = daily_news['news_neg_share'].fillna(0.5)
daily_news['news_sent_std'] = daily_news['news_sent_std'].fillna(0.0)
daily_news['news_polarity'] = daily_news['news_pos_share'] - daily_news['news_neg_share']
daily_news['news_ma_7'] = daily_news['news_sent_compound'].rolling(7, min_periods=1).mean()
daily_news['news_momentum'] = daily_news['news_sent_compound'] - daily_news['news_ma_7']
daily_news['news_count_ma7'] = daily_news['news_count'].rolling(7, min_periods=1).mean()
daily_news['news_volume_surge'] = daily_news['news_count'] / (daily_news['news_count_ma7'] + 1e-6)

daily_news.to_csv(NEWS_FEATURES_OUT, index=False)

features = pd.read_csv(f'outputs/{COIN_NAME.lower()}_features.csv', parse_dates=['date'])
features = features.drop(columns=[c for c in features.columns if c.startswith('news_')], errors='ignore')
NEWS_FEATURES = ['news_count','news_sent_pos','news_sent_neu','news_sent_neg',
                 'news_sent_compound','news_pos_share','news_neg_share','news_polarity',
                 'news_sent_std','news_ma_7','news_momentum','news_volume_surge']
merged = features.merge(daily_news[['date'] + NEWS_FEATURES], on='date', how='left')
for col in NEWS_FEATURES:
    if col in ['news_sent_pos','news_sent_neu','news_sent_neg']:
        merged[col] = merged[col].fillna(1/3)
    elif col in ['news_pos_share','news_neg_share']:
        merged[col] = merged[col].fillna(0.5)
    elif col in ['news_sent_compound','news_polarity','news_ma_7','news_momentum','news_sent_std']:
        merged[col] = merged[col].fillna(0.0)
    else:
        merged[col] = merged[col].fillna(0)
merged.to_csv(f'outputs/{COIN_NAME.lower()}_features.csv', index=False)
print(f"News merged: {len(merged)} rows × {len(merged.columns)} columns")

Restored news cache: 228,446 entries
Loading CoinDesk news...
  Total: 229,172
  After ETH filter: 118,636
  After spam filter: 118,390
Total: 118,101 | Cached: 118,101 | Pending: 0

News class balance:
  Positive articles: 62195 (52.7%)
  Negative articles: 31916 (27.0%)
News merged: 2916 rows × 75 columns


In [ ]:
import pandas as pd, numpy as np, os, re, hashlib, torch, shutil, time

NEWS_FEATURES_OUT = 'outputs/news/news_daily_features.csv'
NEWS_CACHE = 'outputs/news/_news_finbert_cache.csv'
DRIVE_NEWS_CACHE = f'{DRIVE_DIR}/_news_finbert_cache.csv'

if os.path.exists(DRIVE_NEWS_CACHE):
    shutil.copy(DRIVE_NEWS_CACHE, NEWS_CACHE)
    print(f"Restored news cache: {len(pd.read_csv(NEWS_CACHE)):,} entries")

print("Loading CoinDesk news...")
news_df = pd.read_csv(NEWS_CSV)
print(f"  Total: {len(news_df):,}")

news_df['date'] = pd.to_datetime(news_df['published_on'], errors='coerce').dt.normalize()
news_df = news_df.dropna(subset=['date']).reset_index(drop=True)
news_df = news_df[(news_df['date'] >= '2019-01-01') & (news_df['date'] <= '2025-12-31')].reset_index(drop=True)

target_tag = 'ETH'
news_df['tags'] = news_df['tags'].fillna('')
news_df['categories'] = news_df['categories'].fillna('')
news_df['title'] = news_df['title'].fillna('')
news_df['body'] = news_df['body'].fillna('')

def is_relevant(row):
    text = f"{row['tags']} {row['categories']} {row['title']}".upper()
    return target_tag in text

news_df = news_df[news_df.apply(is_relevant, axis=1)].reset_index(drop=True)
print(f"  After ETH filter: {len(news_df):,}")

spam_keywords = ['presale', 'sponsored', 'disclaimer', 'lightchain', 'jump into',
                 "don't miss out", 'token presale', 'whitepaper']
def is_spam(row):
    text = f"{row['title']} {row['body'][:500]}".lower()
    return sum(kw in text for kw in spam_keywords) >= 2

news_df = news_df[~news_df.apply(is_spam, axis=1)].reset_index(drop=True)
print(f"  After spam filter: {len(news_df):,}")

def clean_html(text):
    text = re.sub(r'&apos;', "'", str(text))
    text = re.sub(r'&quot;', '"', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'http\S+', '', text)
    return re.sub(r'\s+', ' ', text).strip()

news_df['title_clean'] = news_df['title'].apply(clean_html)
news_df['body_clean'] = news_df['body'].apply(lambda x: clean_html(x)[:500])
news_df['combined_text'] = news_df['title_clean'] + ". " + news_df['body_clean']
news_df = news_df[news_df['combined_text'].str.len() > 30].reset_index(drop=True)

def make_id(row):
    raw = f"{row['date']}_{row['source']}_{str(row['title_clean'])[:100]}"
    return hashlib.md5(raw.encode()).hexdigest()[:16]

news_df['article_id'] = news_df.apply(make_id, axis=1)
news_df = news_df.drop_duplicates('article_id').reset_index(drop=True)

if os.path.exists(NEWS_CACHE):
    cache = pd.read_csv(NEWS_CACHE)
    if 'article_id' in cache.columns and 'news_pos' in cache.columns:
        news_df = news_df.merge(cache[['article_id','news_pos','news_neu','news_neg']], on='article_id', how='left')
    else:
        for c in ['news_pos','news_neu','news_neg']: news_df[c] = np.nan
else:
    for c in ['news_pos','news_neu','news_neg']: news_df[c] = np.nan

pending = news_df[news_df['news_pos'].isna()].reset_index(drop=True)
print(f"Total: {len(news_df):,} | Cached: {len(news_df)-len(pending):,} | Pending: {len(pending):,}")

if len(pending) > 0:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    print("Scoring with FinBERT...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    tok = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert').to(device).eval()
    BATCH = 64 if torch.cuda.is_available() else 16
    results = []
    t_start = time.time()
    with torch.no_grad():
        for start in range(0, len(pending), BATCH):
            batch_text = pending['combined_text'].iloc[start:start+BATCH].tolist()
            batch_ids = pending['article_id'].iloc[start:start+BATCH].tolist()
            try:
                enc = tok(batch_text, padding=True, truncation=True, max_length=256, return_tensors='pt').to(device)
                probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
                for i, aid in enumerate(batch_ids):
                    results.append({'article_id': aid, 'news_pos': float(probs[i,0]),
                                    'news_neg': float(probs[i,1]), 'news_neu': float(probs[i,2])})
            except: continue
            if (start // BATCH) % 50 == 0:
                elapsed = time.time() - t_start
                rate = (start + len(batch_text)) / max(elapsed, 1)
                eta = (len(pending) - start - len(batch_text)) / max(rate, 1) / 60
                print(f"   {start+len(batch_text):,}/{len(pending):,} | {rate:.0f}/s | ETA: {eta:.1f} min")
            if len(results) >= 2000:
                pp = pd.DataFrame(results)
                if os.path.exists(NEWS_CACHE):
                    pp = pd.concat([pd.read_csv(NEWS_CACHE), pp]).drop_duplicates('article_id')
                pp.to_csv(NEWS_CACHE, index=False)
                shutil.copy(NEWS_CACHE, DRIVE_NEWS_CACHE)
                results = []
    if results:
        pp = pd.DataFrame(results)
        if os.path.exists(NEWS_CACHE):
            pp = pd.concat([pd.read_csv(NEWS_CACHE), pp]).drop_duplicates('article_id')
        pp.to_csv(NEWS_CACHE, index=False)
    shutil.copy(NEWS_CACHE, DRIVE_NEWS_CACHE)
    del model, tok
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    news_df = news_df.drop(columns=['news_pos','news_neu','news_neg'], errors='ignore')
    news_df = news_df.merge(pd.read_csv(NEWS_CACHE), on='article_id', how='left')

news_df['news_compound'] = news_df['news_pos'] - news_df['news_neg']

print(f"\nBEFORE BALANCING:")
old_pos = (news_df['news_compound'] > 0.1).sum()
old_neg = (news_df['news_compound'] < -0.1).sum()
old_neu = ((news_df['news_compound'] >= -0.1) & (news_df['news_compound'] <= 0.1)).sum()
total = len(news_df)
print(f"  Positive: {old_pos} ({100*old_pos/total:.1f}%)")
print(f"  Negative: {old_neg} ({100*old_neg/total:.1f}%)")
print(f"  Neutral:  {old_neu} ({100*old_neu/total:.1f}%)")
print(f"  Pos:Neg ratio = {old_pos/max(old_neg,1):.2f}:1")
print(f"  Mean compound: {news_df['news_compound'].mean():.4f}")
print(f"  Std compound:  {news_df['news_compound'].std():.4f}")

print(f"\nFix 1: Percentile-based class assignment (forces 33/33/33 balance)")
threshold_pos = news_df['news_compound'].quantile(0.66)
threshold_neg = news_df['news_compound'].quantile(0.33)
print(f"  Pos threshold (66th percentile): {threshold_pos:.3f}")
print(f"  Neg threshold (33rd percentile): {threshold_neg:.3f}")

news_df['sentiment_class'] = 'neu'
news_df.loc[news_df['news_compound'] > threshold_pos, 'sentiment_class'] = 'pos'
news_df.loc[news_df['news_compound'] < threshold_neg, 'sentiment_class'] = 'neg'

print(f"\nAFTER PERCENTILE-BASED REBALANCING:")
new_pos = (news_df['sentiment_class']=='pos').sum()
new_neg = (news_df['sentiment_class']=='neg').sum()
new_neu = (news_df['sentiment_class']=='neu').sum()
print(f"  Positive: {new_pos} ({100*new_pos/total:.1f}%)")
print(f"  Negative: {new_neg} ({100*new_neg/total:.1f}%)")
print(f"  Neutral:  {new_neu} ({100*new_neu/total:.1f}%)")

print(f"\nFix 2: Inverse-frequency balance weights for daily aggregation")
news_df['balance_weight'] = 1.0
news_df.loc[news_df['sentiment_class']=='pos', 'balance_weight'] = total / (3 * max(new_pos, 1))
news_df.loc[news_df['sentiment_class']=='neg', 'balance_weight'] = total / (3 * max(new_neg, 1))
news_df.loc[news_df['sentiment_class']=='neu', 'balance_weight'] = total / (3 * max(new_neu, 1))
print(f"  Pos weight: {total/(3*max(new_pos,1)):.3f}")
print(f"  Neg weight: {total/(3*max(new_neg,1)):.3f}")
print(f"  Neu weight: {total/(3*max(new_neu,1)):.3f}")

news_df['is_pos'] = (news_df['sentiment_class']=='pos').astype(int)
news_df['is_neg'] = (news_df['sentiment_class']=='neg').astype(int)
news_df['is_neu'] = (news_df['sentiment_class']=='neu').astype(int)

print(f"\nFix 3: Computing balanced daily aggregations with new features")

def agg_daily(g):
    w = g['balance_weight'].values
    ws = w.sum()
    if ws == 0:
        w = np.ones_like(w); ws = len(w)

    news_pos_balanced = (g['news_pos'] * w).sum() / ws
    news_neu_balanced = (g['news_neu'] * w).sum() / ws
    news_neg_balanced = (g['news_neg'] * w).sum() / ws
    news_compound_balanced = news_pos_balanced - news_neg_balanced

    pos_count = g['is_pos'].sum()
    neg_count = g['is_neg'].sum()
    neu_count = g['is_neu'].sum()
    n = len(g)

    return pd.Series({
        'news_count': n,
        'news_sent_pos': news_pos_balanced,
        'news_sent_neu': news_neu_balanced,
        'news_sent_neg': news_neg_balanced,
        'news_sent_compound': news_compound_balanced,
        'news_pos_share': pos_count / max(n, 1),
        'news_neg_share': neg_count / max(n, 1),
        'news_neu_share': neu_count / max(n, 1),
        'news_polarity_ratio': (pos_count - neg_count) / max(pos_count + neg_count, 1),
        'news_sent_std': g['news_compound'].std() if n > 1 else 0,
        'news_extreme_pos': (g['news_pos'] > 0.7).sum() / max(n, 1),
        'news_extreme_neg': (g['news_neg'] > 0.5).sum() / max(n, 1),
        'news_disagreement': g['news_compound'].std() if n > 1 else 0,
    })

daily_news = news_df.groupby('date').apply(agg_daily).reset_index()
price_df = pd.read_csv(f'outputs/{COIN_NAME.lower()}_features.csv', parse_dates=['date'])
all_dates = pd.DataFrame({'date': sorted(price_df['date'].unique())})
daily_news = all_dates.merge(daily_news, on='date', how='left')

print(f"\nDaily news statistics (after balancing):")
print(f"  Mean compound:        {daily_news['news_sent_compound'].mean():.4f}")
print(f"  Std compound:         {daily_news['news_sent_compound'].std():.4f}")
print(f"  Range compound:       [{daily_news['news_sent_compound'].min():+.3f}, {daily_news['news_sent_compound'].max():+.3f}]")
print(f"  Mean pos_share:       {daily_news['news_pos_share'].mean():.3f}")
print(f"  Mean neg_share:       {daily_news['news_neg_share'].mean():.3f}")
print(f"  Mean polarity_ratio:  {daily_news['news_polarity_ratio'].mean():.4f}")
print(f"  Std polarity_ratio:   {daily_news['news_polarity_ratio'].std():.4f}")

for col in ['news_sent_pos','news_sent_neu','news_sent_neg','news_sent_compound',
            'news_pos_share','news_neg_share','news_neu_share','news_polarity_ratio',
            'news_extreme_pos','news_extreme_neg','news_disagreement']:
    vals = daily_news[col].values.copy()
    last, streak = np.nan, 0
    for i in range(len(vals)):
        if np.isnan(vals[i]):
            if not np.isnan(last):
                streak += 1
                decay = 0.7 ** streak
                if 'compound' in col or 'polarity' in col or 'disagreement' in col:
                    vals[i] = last * decay
                else:
                    vals[i] = last * decay + (1 - decay) * 0.33
        else:
            last, streak = vals[i], 0
    daily_news[col] = vals

daily_news['news_count'] = daily_news['news_count'].fillna(0)
fill_defaults = {
    'news_sent_pos': 1/3, 'news_sent_neu': 1/3, 'news_sent_neg': 1/3,
    'news_sent_compound': 0.0, 'news_pos_share': 0.33, 'news_neg_share': 0.33,
    'news_neu_share': 0.33, 'news_polarity_ratio': 0.0,
    'news_sent_std': 0.0, 'news_extreme_pos': 0.0, 'news_extreme_neg': 0.0,
    'news_disagreement': 0.0
}
for col, default in fill_defaults.items():
    daily_news[col] = daily_news[col].fillna(default)

daily_news['news_ma_3'] = daily_news['news_sent_compound'].rolling(3, min_periods=1).mean()
daily_news['news_ma_7'] = daily_news['news_sent_compound'].rolling(7, min_periods=1).mean()
daily_news['news_momentum'] = daily_news['news_sent_compound'] - daily_news['news_ma_7']
daily_news['news_polarity_momentum'] = daily_news['news_polarity_ratio'] - daily_news['news_polarity_ratio'].rolling(7, min_periods=1).mean()
daily_news['news_count_ma7'] = daily_news['news_count'].rolling(7, min_periods=1).mean()
daily_news['news_volume_surge'] = daily_news['news_count'] / (daily_news['news_count_ma7'] + 1e-6)

m = daily_news['news_sent_compound'].rolling(30, min_periods=5).mean()
s = daily_news['news_sent_compound'].rolling(30, min_periods=5).std()
daily_news['news_zscore'] = ((daily_news['news_sent_compound']-m)/(s+1e-6)).fillna(0)

daily_news.to_csv(NEWS_FEATURES_OUT, index=False)

print(f"\nFinal news feature statistics (variance check):")
for col in ['news_sent_compound', 'news_polarity_ratio', 'news_pos_share', 'news_neg_share',
            'news_disagreement', 'news_zscore', 'news_polarity_momentum', 'news_volume_surge']:
    print(f"  {col:28s}: std={daily_news[col].std():.4f}  range=[{daily_news[col].min():+.3f}, {daily_news[col].max():+.3f}]")

features = pd.read_csv(f'outputs/{COIN_NAME.lower()}_features.csv', parse_dates=['date'])
features = features.drop(columns=[c for c in features.columns if c.startswith('news_')], errors='ignore')

NEWS_FEATURES = ['news_count','news_sent_pos','news_sent_neu','news_sent_neg',
                 'news_sent_compound','news_pos_share','news_neg_share','news_neu_share',
                 'news_polarity_ratio','news_sent_std','news_extreme_pos','news_extreme_neg',
                 'news_disagreement','news_ma_3','news_ma_7','news_momentum',
                 'news_polarity_momentum','news_volume_surge','news_zscore']

merged = features.merge(daily_news[['date'] + NEWS_FEATURES], on='date', how='left')

for col in NEWS_FEATURES:
    if col in ['news_sent_pos','news_sent_neu','news_sent_neg']:
        merged[col] = merged[col].fillna(1/3)
    elif col in ['news_pos_share','news_neg_share','news_neu_share']:
        merged[col] = merged[col].fillna(0.33)
    elif col in ['news_sent_compound','news_polarity_ratio','news_ma_3','news_ma_7',
                 'news_momentum','news_polarity_momentum','news_disagreement',
                 'news_extreme_pos','news_extreme_neg','news_sent_std','news_zscore']:
        merged[col] = merged[col].fillna(0.0)
    else:
        merged[col] = merged[col].fillna(0)

merged.to_csv(f'outputs/{COIN_NAME.lower()}_features.csv', index=False)
print(f"\nNews merged: {len(merged)} rows × {len(merged.columns)} columns")
print(f"Total news features: {len(NEWS_FEATURES)} (was 12, now {len(NEWS_FEATURES)} with new balance-aware features)")

Restored news cache: 228,446 entries
Loading CoinDesk news...
  Total: 229,172
  After ETH filter: 118,636
  After spam filter: 118,390
Total: 118,101 | Cached: 118,101 | Pending: 0

BEFORE BALANCING:
  Positive: 62195 (52.7%)
  Negative: 31916 (27.0%)
  Neutral:  23990 (20.3%)
  Pos:Neg ratio = 1.95:1
  Mean compound: 0.1091
  Std compound:  0.6026

Fix 1: Percentile-based class assignment (forces 33/33/33 balance)
  Pos threshold (66th percentile): 0.388
  Neg threshold (33rd percentile): 0.008

AFTER PERCENTILE-BASED REBALANCING:
  Positive: 40154 (34.0%)
  Negative: 38973 (33.0%)
  Neutral:  38974 (33.0%)

Fix 2: Inverse-frequency balance weights for daily aggregation
  Pos weight: 0.980
  Neg weight: 1.010
  Neu weight: 1.010

Fix 3: Computing balanced daily aggregations with new features

Daily news statistics (after balancing):
  Mean compound:        0.0808
  Std compound:         0.1951
  Range compound:       [-0.625, +0.618]
  Mean pos_share:       0.318
  Mean neg_share:   

In [ ]:
NEWS_FEATURES = ['news_count','news_sent_pos','news_sent_neu','news_sent_neg',
                 'news_sent_compound','news_pos_share','news_neg_share','news_neu_share',
                 'news_polarity_ratio','news_sent_std','news_extreme_pos','news_extreme_neg',
                 'news_disagreement','news_ma_3','news_ma_7','news_momentum',
                 'news_polarity_momentum','news_volume_surge','news_zscore']

In [ ]:
import pandas as pd, numpy as np, os, pickle
from sklearn.preprocessing import QuantileTransformer

LOOKBACK = 60
PURGE_DAYS = 14
DATA_START = '2019-10-29'
TEST_END = '2025-02-01'
MIN_NEWS_PER_DAY = 1

print("LOADING DATA")
features_file = f'outputs/{COIN_NAME.lower()}_features.csv'
features = pd.read_csv(features_file, parse_dates=['date'])

onchain_file = f'outputs/onchain/{COIN_NAME.lower()}_onchain.csv'
ONCHAIN_FEATURES = ['vol_to_mcap','vol_to_mcap_ma7','vol_to_mcap_change','mcap_zscore_30',
                    'volume_surge','volume_zscore_30','price_vol_divergence',
                    'taker_buy_ratio','taker_buy_ratio_ma7','trades_per_day_zscore']
if os.path.exists(onchain_file):
    onchain = pd.read_csv(onchain_file, parse_dates=['date'])
    features = features.drop(columns=[c for c in ONCHAIN_FEATURES if c in features.columns], errors='ignore')
    features = features.merge(onchain[['date'] + ONCHAIN_FEATURES], on='date', how='left')
    for col in ONCHAIN_FEATURES:
        features[col] = features[col].ffill().fillna(0)

news_file = 'outputs/news/news_daily_features.csv'
NEWS_FEATURES = ['news_count','news_sent_pos','news_sent_neu','news_sent_neg',
                 'news_sent_compound','news_pos_share','news_neg_share','news_neu_share',
                 'news_polarity_ratio','news_sent_std','news_extreme_pos','news_extreme_neg',
                 'news_disagreement','news_ma_3','news_ma_7','news_momentum',
                 'news_polarity_momentum','news_volume_surge','news_zscore']
if os.path.exists(news_file):
    news = pd.read_csv(news_file, parse_dates=['date'])
    features = features.drop(columns=[c for c in NEWS_FEATURES if c in features.columns], errors='ignore')
    features = features.merge(news[['date'] + NEWS_FEATURES], on='date', how='left')
    for col in NEWS_FEATURES:
        if col in ['news_sent_pos','news_sent_neu','news_sent_neg']:
            features[col] = features[col].fillna(1/3)
        elif col in ['news_pos_share','news_neg_share']:
            features[col] = features[col].fillna(0.5)
        elif col in ['news_sent_compound','news_polarity','news_ma_7','news_momentum','news_sent_std']:
            features[col] = features[col].fillna(0.0)
        else:
            features[col] = features[col].fillna(0)

features.to_csv(features_file, index=False)

sent_file = f'outputs/{COIN_NAME.lower()}_sentiment_daily.csv'
sent = pd.read_csv(sent_file, parse_dates=['date'])
df = features.merge(sent, on='date', how='left')
df = df[df['date'] >= DATA_START].reset_index(drop=True)

HARD_FEATURES = ['sma_10_ratio','sma_20_ratio','sma_50_ratio','ema_12_ratio','ema_26_ratio',
                 'sma_cross_20_50','ema_cross_12_26',
                 'macd_norm','macd_signal_norm','macd_hist_norm','adx_14',
                 'rsi_7','rsi_14','rsi_21','rsi_divergence',
                 'stoch_k','stoch_d','stoch_kd_diff',
                 'roc_3','roc_7','roc_14','mfi_14',
                 'bb_width','bb_pctb','atr_pct',
                 'hist_vol_10','hist_vol_20','vol_of_vol',
                 'cmf_20','vol_sma_ratio','vol_zscore','vol_acceleration',
                 'log_ret_1','log_ret_3','log_ret_7','log_ret_14',
                 'ret_skew_14','ret_kurt_14',
                 'hl_range','close_position','ret_zscore_20']
SOFT_FEATURES = ['sent_pos','sent_neu','sent_neg','sent_compound',
                 'sent_pos_share','sent_neg_share','sent_neu_share','sent_polarity_ratio',
                 'sent_disagreement','sent_extreme_pos','sent_extreme_neg',
                 'sent_ma_3','sent_ma_7','sent_std_7','sent_momentum',
                 'sent_zscore','tweet_count_ratio','polarity_momentum']
HARD_FEATURES_FULL = HARD_FEATURES + ONCHAIN_FEATURES + NEWS_FEATURES

missing = [f for f in HARD_FEATURES_FULL + SOFT_FEATURES if f not in df.columns]
if missing:
    print(f"MISSING: {missing}")
    raise ValueError(f"Missing {len(missing)}")
print(f"\nFeature breakdown:")
print(f"  Technical: {len(HARD_FEATURES)} (all RATIOS - regime-robust)")
print(f"  On-chain:  {len(ONCHAIN_FEATURES)}")
print(f"  News:      {len(NEWS_FEATURES)}")
print(f"  Twitter:   {len(SOFT_FEATURES)}")
print(f"  Hard:      {len(HARD_FEATURES_FULL)}")
print(f"  Soft:      {len(SOFT_FEATURES)}")

df_news_only = df[df['news_count'] >= MIN_NEWS_PER_DAY].reset_index(drop=True)
print(f"\nDays with news: {len(df_news_only)}/{len(df)} ({100*len(df_news_only)/len(df):.1f}%)")

print("\nREGIME-BALANCED SPLITS")
df_seq = df_news_only.copy()

test_cutoff = pd.Timestamp(TEST_END)
test_start_actual = test_cutoff - pd.Timedelta(days=180)
trainval_end = test_start_actual - pd.Timedelta(days=PURGE_DAYS)
val_size_days = 240
val_start_actual = trainval_end - pd.Timedelta(days=val_size_days)

train_df = df_seq[df_seq['date'] < val_start_actual].reset_index(drop=True)
val_df = df_seq[(df_seq['date'] >= val_start_actual) & (df_seq['date'] < trainval_end)].reset_index(drop=True)
test_df = df_seq[(df_seq['date'] >= test_start_actual) & (df_seq['date'] <= test_cutoff)].reset_index(drop=True)

print(f"TRAIN: {len(train_df)} ({train_df['date'].min().date()} to {train_df['date'].max().date()})")
print(f"VAL:   {len(val_df)} ({val_df['date'].min().date()} to {val_df['date'].max().date()})")
print(f"TEST:  {len(test_df)} ({test_df['date'].min().date()} to {test_df['date'].max().date()})")

print("\nUP% by horizon (checking regime alignment):")
for h in [1, 7, 30]:
    tc = f'target_{h}d'
    tr = train_df[tc].dropna()
    vl = val_df[tc].dropna()
    te = test_df[tc].dropna()
    diff = abs(vl.mean() - te.mean()) * 100
    flag = " <-- MISMATCH" if diff > 10 else ""
    print(f"  {h:2d}d: train={100*tr.mean():.1f}%  val={100*vl.mean():.1f}%  test={100*te.mean():.1f}%{flag}")

hard_scaler = QuantileTransformer(n_quantiles=min(500, len(train_df)), output_distribution='normal', random_state=42)
soft_scaler = QuantileTransformer(n_quantiles=min(500, len(train_df)), output_distribution='normal', random_state=42)
hard_scaler.fit(train_df[HARD_FEATURES_FULL])
soft_scaler.fit(train_df[SOFT_FEATURES])

def scale(d):
    d = d.copy()
    d[HARD_FEATURES_FULL] = hard_scaler.transform(d[HARD_FEATURES_FULL])
    d[SOFT_FEATURES] = soft_scaler.transform(d[SOFT_FEATURES])
    return d

df_full_scaled = df.copy()
df_full_scaled[HARD_FEATURES_FULL] = hard_scaler.transform(df_full_scaled[HARD_FEATURES_FULL])
df_full_scaled[SOFT_FEATURES] = soft_scaler.transform(df_full_scaled[SOFT_FEATURES])

train_s = scale(train_df); val_s = scale(val_df); test_s = scale(test_df)

def make_seq(dframe, ctx, tc, rc):
    Xh, Xs, y, dates, rets = [], [], [], [], []
    ctx = ctx.sort_values('date').reset_index(drop=True)
    for _, row in dframe.iterrows():
        idx = ctx.index[ctx['date'] == row['date']]
        if len(idx) == 0: continue
        idx = idx[0]
        if idx < LOOKBACK - 1: continue
        if pd.isna(row[tc]): continue
        w = ctx.iloc[idx-LOOKBACK+1 : idx+1]
        Xh.append(w[HARD_FEATURES_FULL].values)
        Xs.append(w[SOFT_FEATURES].values)
        y.append(int(row[tc])); dates.append(row['date']); rets.append(float(row[rc]))
    return (np.array(Xh, dtype=np.float32), np.array(Xs, dtype=np.float32),
            np.array(y, dtype=np.int64), np.array(dates), np.array(rets, dtype=np.float32))

def make_flat(d, tc, rc):
    X = d[HARD_FEATURES_FULL + SOFT_FEATURES].values.astype(np.float32)
    y = d[tc].values; r = d[rc].values.astype(np.float32); dd = d['date'].values
    keep = ~pd.isna(y)
    return X[keep], y[keep].astype(np.int64), dd[keep], r[keep]

print("\nBuilding sequences...")
for h in [1, 7, 30]:
    tc = f'target_{h}d'; rc = f'return_{h}d'
    Xh_tr, Xs_tr, y_tr, d_tr, r_tr = make_seq(train_s, df_full_scaled, tc, rc)
    Xh_v, Xs_v, y_v, d_v, r_v = make_seq(val_s, df_full_scaled, tc, rc)
    Xh_te, Xs_te, y_te, d_te, r_te = make_seq(test_s, df_full_scaled, tc, rc)
    Xf_tr, yf_tr, df_tr, rf_tr = make_flat(train_s, tc, rc)
    Xf_v, yf_v, df_v, rf_v = make_flat(val_s, tc, rc)
    Xf_te, yf_te, df_te, rf_te = make_flat(test_s, tc, rc)
    print(f"{h}d: train{Xh_tr.shape} val{Xh_v.shape} test{Xh_te.shape} UP%: tr={100*(y_tr==1).mean():.1f} v={100*(y_v==1).mean():.1f} te={100*(y_te==1).mean():.1f}")

    np.savez_compressed(f'outputs/sequences/sequences_{h}d.npz',
        X_hard_train=Xh_tr, X_soft_train=Xs_tr, y_train=y_tr, d_train=d_tr, r_train=r_tr,
        X_hard_val=Xh_v, X_soft_val=Xs_v, y_val=y_v, d_val=d_v, r_val=r_v,
        X_hard_test=Xh_te, X_soft_test=Xs_te, y_test=y_te, d_test=d_te, r_test=r_te,
        X_flat_train=Xf_tr, y_flat_train=yf_tr, d_flat_train=df_tr, r_flat_train=rf_tr,
        X_flat_val=Xf_v, y_flat_val=yf_v, d_flat_val=df_v, r_flat_val=rf_v,
        X_flat_test=Xf_te, y_flat_test=yf_te, d_flat_test=df_te, r_flat_test=rf_te)

with open('outputs/sequences/scalers.pkl', 'wb') as f:
    pickle.dump({'hard': hard_scaler, 'soft': soft_scaler,
                 'hard_features': HARD_FEATURES_FULL, 'soft_features': SOFT_FEATURES}, f)
print("\nSequences saved")

LOADING DATA

Feature breakdown:
  Technical: 41 (all RATIOS - regime-robust)
  On-chain:  10
  News:      19
  Twitter:   18
  Hard:      70
  Soft:      18

Days with news: 1923/2300 (83.6%)

REGIME-BALANCED SPLITS
TRAIN: 1488 (2019-10-29 to 2023-11-24)
VAL:   240 (2023-11-25 to 2024-07-21)
TEST:  181 (2024-08-05 to 2025-02-01)

UP% by horizon (checking regime alignment):
   1d: train=51.9%  val=53.8%  test=52.5%
   7d: train=53.4%  val=53.3%  test=51.9%
  30d: train=57.8%  val=49.2%  test=43.6%

Building sequences...
1d: train(1429, 60, 70) val(240, 60, 70) test(181, 60, 70) UP%: tr=52.4 v=53.8 te=52.5
7d: train(1429, 60, 70) val(240, 60, 70) test(181, 60, 70) UP%: tr=54.5 v=53.3 te=51.9
30d: train(1429, 60, 70) val(240, 60, 70) test(181, 60, 70) UP%: tr=59.1 v=49.2 te=43.6

Sequences saved


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()*(-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class GaussianNoise(nn.Module):
    def __init__(self, sigma=0.05):
        super().__init__()
        self.sigma = sigma
    def forward(self, x):
        if self.training and self.sigma > 0:
            return x + torch.randn_like(x) * self.sigma
        return x

class AttentionLSTM(nn.Module):
    def __init__(self, n_features, hidden=64, n_layers=2, n_classes=2, dropout=0.5):
        super().__init__()
        self.input_noise = GaussianNoise(0.05)
        self.input_drop = nn.Dropout(0.2)
        self.lstm = nn.LSTM(n_features, hidden, n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.attn = nn.Sequential(nn.Linear(hidden*2, hidden), nn.Tanh(),
                                   nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.fc = nn.Sequential(
            nn.LayerNorm(hidden*2), nn.Dropout(dropout),
            nn.Linear(hidden*2, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, n_classes))
    def forward(self, x):
        x = self.input_noise(self.input_drop(x))
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (out*w).sum(dim=1)
        return self.fc(ctx), w.squeeze(-1)

class TransformerPrice(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, n_layers=2, n_classes=2, dropout=0.4):
        super().__init__()
        self.input_noise = GaussianNoise(0.05)
        self.input_drop = nn.Dropout(0.2)
        self.proj = nn.Linear(n_features, d_model)
        self.pe = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model*2, dropout,
                                            batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.pool = nn.Linear(d_model, 1)
        self.fc = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout),
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, n_classes))
    def forward(self, x):
        x = self.input_noise(self.input_drop(x))
        h = self.pe(self.proj(x))
        h = self.encoder(h)
        w = torch.softmax(self.pool(h), dim=1)
        ctx = (h*w).sum(dim=1)
        return self.fc(ctx), w.squeeze(-1)

class CrossAttentionFusion(nn.Module):
    def __init__(self, n_hard, n_soft, d_model=64, nhead=4, n_layers=2, n_classes=2, dropout=0.4):
        super().__init__()
        self.hard_noise = GaussianNoise(0.05)
        self.soft_noise = GaussianNoise(0.05)
        self.hard_drop = nn.Dropout(0.2)
        self.soft_drop = nn.Dropout(0.2)
        self.hard_proj = nn.Linear(n_hard, d_model)
        self.soft_proj = nn.Linear(n_soft, d_model)
        self.pe_hard = PositionalEncoding(d_model)
        self.pe_soft = PositionalEncoding(d_model)
        hl = nn.TransformerEncoderLayer(d_model, nhead, d_model*2, dropout, batch_first=True, activation='gelu')
        sl = nn.TransformerEncoderLayer(d_model, nhead, d_model*2, dropout, batch_first=True, activation='gelu')
        self.hard_encoder = nn.TransformerEncoder(hl, n_layers)
        self.soft_encoder = nn.TransformerEncoder(sl, n_layers)
        self.cross = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.pool_h = nn.Linear(d_model, 1)
        self.pool_s = nn.Linear(d_model, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model*2), nn.Dropout(dropout),
            nn.Linear(d_model*2, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, n_classes))
    def forward(self, x_hard, x_soft):
        x_hard = self.hard_noise(self.hard_drop(x_hard))
        x_soft = self.soft_noise(self.soft_drop(x_soft))
        H = self.pe_hard(self.hard_proj(x_hard))
        S = self.pe_soft(self.soft_proj(x_soft))
        H = self.hard_encoder(H); S = self.soft_encoder(S)
        H_c, _ = self.cross(H, S, S, need_weights=False)
        H = self.norm(H + H_c)
        wh = torch.softmax(self.pool_h(H), dim=1)
        ws = torch.softmax(self.pool_s(S), dim=1)
        zh = (H*wh).sum(dim=1); zs = (S*ws).sum(dim=1)
        return self.classifier(torch.cat([zh, zs], dim=-1)), {'h':wh.squeeze(-1),'s':ws.squeeze(-1)}

class TCN(nn.Module):
    def __init__(self, n_features, channels=64, n_classes=2, dropout=0.4):
        super().__init__()
        self.input_noise = GaussianNoise(0.05)
        self.input_drop = nn.Dropout(0.2)
        layers = []
        in_ch = n_features
        for d in [1, 2, 4, 8]:
            layers += [
                nn.Conv1d(in_ch, channels, 3, padding=d, dilation=d),
                nn.BatchNorm1d(channels), nn.GELU(), nn.Dropout(dropout)
            ]
            in_ch = channels
        self.tcn = nn.Sequential(*layers)
        self.attn = nn.Sequential(nn.Linear(channels, channels), nn.Tanh(), nn.Linear(channels, 1))
        self.fc = nn.Sequential(
            nn.LayerNorm(channels), nn.Dropout(dropout),
            nn.Linear(channels, channels), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(channels, n_classes))
    def forward(self, x):
        x = self.input_noise(self.input_drop(x))
        h = self.tcn(x.transpose(1, 2)).transpose(1, 2)
        w = torch.softmax(self.attn(h), dim=1)
        ctx = (h*w).sum(dim=1)
        return self.fc(ctx), w.squeeze(-1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5, label_smoothing=0.1):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma; self.ls=label_smoothing
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                             label_smoothing=self.ls, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

print("Models defined: smaller capacity, dropout 0.4-0.5, input noise, layernorm")

Models defined: smaller capacity, dropout 0.4-0.5, input noise, layernorm


In [ ]:
import numpy as np, torch, os, json, random, time, math
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score, balanced_accuracy_score

OUT_DIR = 'outputs/models'
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 80
BATCH = 64
LR = 5e-4
WD = 5e-3
PATIENCE = 12
SEEDS = [42, 123, 2024, 7, 999]

print(f"Device: {DEVICE}")
print(f"Config: epochs={EPOCHS} batch={BATCH} lr={LR} wd={WD} patience={PATIENCE}")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_probs(model, loader, use_soft):
    model.eval()
    all_pr, all_y = [], []
    with torch.no_grad():
        for batch in loader:
            if use_soft:
                xh, xs, y = batch
                xh, xs = xh.to(DEVICE), xs.to(DEVICE)
                out, _ = model(xh, xs)
            else:
                xh, y = batch
                xh = xh.to(DEVICE)
                out, _ = model(xh)
            pr = torch.softmax(out, dim=-1).cpu().numpy()
            all_pr.append(pr); all_y.append(y.numpy())
    return np.concatenate(all_pr), np.concatenate(all_y)

def find_best_threshold(probs_up, y_true):
    best_thr, best_score = 0.5, -1
    for thr in np.arange(0.30, 0.71, 0.01):
        preds = (probs_up > thr).astype(int)
        if preds.sum() == 0 or preds.sum() == len(preds):
            continue
        score = balanced_accuracy_score(y_true, preds)
        if score > best_score:
            best_score = score
            best_thr = thr
    return best_thr, best_score

def detect_collapse(preds):
    if len(preds) == 0:
        return True
    rate1 = (preds == 1).mean()
    return rate1 < 0.05 or rate1 > 0.95

def train_one(model_fn, name, use_soft, horizon):
    print(f"\n{'='*70}\n{name} | {horizon}d\n{'='*70}")
    data = np.load(f'outputs/sequences/sequences_{horizon}d.npz', allow_pickle=True)
    Xh_tr, Xs_tr, y_tr = data['X_hard_train'], data['X_soft_train'], data['y_train']
    Xh_v, Xs_v, y_v = data['X_hard_val'], data['X_soft_val'], data['y_val']
    Xh_te, Xs_te, y_te = data['X_hard_test'], data['X_soft_test'], data['y_test']

    cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_tr)
    cw_t = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

    if use_soft:
        tr_ds = TensorDataset(torch.tensor(Xh_tr), torch.tensor(Xs_tr), torch.tensor(y_tr))
        v_ds = TensorDataset(torch.tensor(Xh_v), torch.tensor(Xs_v), torch.tensor(y_v))
        te_ds = TensorDataset(torch.tensor(Xh_te), torch.tensor(Xs_te), torch.tensor(y_te))
    else:
        tr_ds = TensorDataset(torch.tensor(Xh_tr), torch.tensor(y_tr))
        v_ds = TensorDataset(torch.tensor(Xh_v), torch.tensor(y_v))
        te_ds = TensorDataset(torch.tensor(Xh_te), torch.tensor(y_te))

    tr_l = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, drop_last=True)
    v_l = DataLoader(v_ds, batch_size=BATCH, shuffle=False)
    te_l = DataLoader(te_ds, batch_size=BATCH, shuffle=False)

    seed_results = []
    for seed in SEEDS:
        set_seed(seed)
        model = model_fn().to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        warmup = 5
        def ll(ep):
            if ep < warmup: return (ep+1) / warmup
            prog = (ep - warmup) / max(1, EPOCHS - warmup)
            return 0.5 * (1 + math.cos(math.pi * prog))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)
        crit = FocalLoss(alpha=cw_t, gamma=1.5, label_smoothing=0.1)

        best_auc, best_state, pat = -1, None, 0
        for ep in range(EPOCHS):
            model.train()
            for batch in tr_l:
                if use_soft:
                    xh, xs, y = batch
                    xh, xs, y = xh.to(DEVICE), xs.to(DEVICE), y.to(DEVICE)
                    out, _ = model(xh, xs)
                else:
                    xh, y = batch
                    xh, y = xh.to(DEVICE), y.to(DEVICE)
                    out, _ = model(xh)
                loss = crit(out, y)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                opt.step()
            sched.step()

            v_pr, v_y = get_probs(model, v_l, use_soft)
            try: v_auc = roc_auc_score(v_y, v_pr[:, 1])
            except: v_auc = 0.5

            if v_auc > best_auc:
                best_auc = v_auc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                pat = 0
            else:
                pat += 1

            if ep % 10 == 0 or pat == 0:
                print(f"  seed={seed} ep={ep:3d} val_auc={v_auc:.4f} best={best_auc:.4f}")

            if pat >= PATIENCE:
                break

        model.load_state_dict(best_state)

        v_pr, v_y = get_probs(model, v_l, use_soft)
        thr, val_bal_acc = find_best_threshold(v_pr[:, 1], v_y)

        te_pr, te_y = get_probs(model, te_l, use_soft)
        te_preds = (te_pr[:, 1] > thr).astype(int)

        if detect_collapse(te_preds):
            te_preds_05 = (te_pr[:, 1] > 0.5).astype(int)
            if not detect_collapse(te_preds_05):
                te_preds = te_preds_05
                thr = 0.5

        ta = accuracy_score(te_y, te_preds)
        tba = balanced_accuracy_score(te_y, te_preds)
        tf = f1_score(te_y, te_preds, average='macro', zero_division=0)
        tm = matthews_corrcoef(te_y, te_preds)
        try: tauc = roc_auc_score(te_y, te_pr[:, 1])
        except: tauc = 0.5

        print(f"  seed={seed} thr={thr:.2f} acc={ta:.4f} bal={tba:.4f} mcc={tm:+.3f} auc={tauc:.4f} pred_up%={100*(te_preds==1).mean():.0f}")

        seed_results.append({
            'seed': seed, 'acc': ta, 'bal_acc': tba, 'f1': tf, 'mcc': tm, 'auc': tauc,
            'thr': thr, 'probs': te_pr, 'preds': te_preds, 'y': te_y,
            'val_probs': v_pr, 'val_y': v_y, 'state': best_state
        })

    valid = [r for r in seed_results if not detect_collapse(r['preds'])]
    if len(valid) == 0:
        valid = seed_results
        print(f"  WARNING: all seeds collapsed for {name} {horizon}d")

    avg_probs = np.mean([r['probs'] for r in valid], axis=0)
    avg_val_probs = np.mean([r['val_probs'] for r in valid], axis=0)

    ens_thr, _ = find_best_threshold(avg_val_probs[:, 1], valid[0]['val_y'])
    ens_preds = (avg_probs[:, 1] > ens_thr).astype(int)
    if detect_collapse(ens_preds):
        ens_preds = (avg_probs[:, 1] > 0.5).astype(int)
        ens_thr = 0.5

    ens_y = valid[0]['y']
    ens_acc = accuracy_score(ens_y, ens_preds)
    ens_bal = balanced_accuracy_score(ens_y, ens_preds)
    ens_mcc = matthews_corrcoef(ens_y, ens_preds)
    try: ens_auc = roc_auc_score(ens_y, avg_probs[:, 1])
    except: ens_auc = 0.5

    print(f"  ENSEMBLE: thr={ens_thr:.2f} acc={ens_acc:.4f} bal={ens_bal:.4f} mcc={ens_mcc:+.3f} auc={ens_auc:.4f}")

    key = f'{name}_{horizon}d'
    best_seed_state = max(seed_results, key=lambda x: x['auc'])['state']
    torch.save(best_seed_state, f'{OUT_DIR}/{key}.pt')
    np.savez(f'{OUT_DIR}/{key}_preds.npz',
             probs=avg_probs, preds=ens_preds, y=ens_y,
             val_probs=avg_val_probs, val_y=valid[0]['val_y'],
             threshold=ens_thr)

    return {'model': name, 'horizon': f'{horizon}d',
            'acc_mean': float(np.mean([r['acc'] for r in valid])),
            'acc_std': float(np.std([r['acc'] for r in valid])),
            'bal_acc_mean': float(np.mean([r['bal_acc'] for r in valid])),
            'mcc_mean': float(np.mean([r['mcc'] for r in valid])),
            'auc_mean': float(np.mean([r['auc'] for r in valid])),
            'ensemble_acc': float(ens_acc),
            'ensemble_bal_acc': float(ens_bal),
            'ensemble_mcc': float(ens_mcc),
            'ensemble_auc': float(ens_auc),
            'ensemble_threshold': float(ens_thr),
            'n_valid_seeds': len(valid)}

def train_xgb(horizon):
    import xgboost as xgb
    print(f"\n{'='*70}\nXGBoost | {horizon}d\n{'='*70}")
    data = np.load(f'outputs/sequences/sequences_{horizon}d.npz', allow_pickle=True)
    Xf_tr, yf_tr = data['X_flat_train'], data['y_flat_train']
    Xf_v, yf_v = data['X_flat_val'], data['y_flat_val']
    Xf_te, yf_te = data['X_flat_test'], data['y_flat_test']

    sw = compute_sample_weight('balanced', yf_tr)
    seed_results = []
    for seed in SEEDS:
        clf = xgb.XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.03,
            subsample=0.7, colsample_bytree=0.6,
            min_child_weight=10, gamma=0.5, reg_lambda=2.0, reg_alpha=0.5,
            objective='binary:logistic', eval_metric='auc',
            early_stopping_rounds=20, random_state=seed, n_jobs=-1, verbosity=0)
        clf.fit(Xf_tr, yf_tr, sample_weight=sw, eval_set=[(Xf_v, yf_v)], verbose=False)

        v_probs = clf.predict_proba(Xf_v)
        thr, _ = find_best_threshold(v_probs[:, 1], yf_v)
        t_probs = clf.predict_proba(Xf_te)
        t_preds = (t_probs[:, 1] > thr).astype(int)
        if detect_collapse(t_preds):
            t_preds = (t_probs[:, 1] > 0.5).astype(int)
            thr = 0.5
        acc = accuracy_score(yf_te, t_preds)
        bal = balanced_accuracy_score(yf_te, t_preds)
        mcc = matthews_corrcoef(yf_te, t_preds)
        try: auc = roc_auc_score(yf_te, t_probs[:, 1])
        except: auc = 0.5
        print(f"  seed={seed} thr={thr:.2f} acc={acc:.4f} bal={bal:.4f} mcc={mcc:+.3f} auc={auc:.4f}")
        seed_results.append({
            'seed': seed, 'acc': acc, 'bal_acc': bal, 'mcc': mcc, 'auc': auc,
            'thr': thr, 'probs': t_probs, 'preds': t_preds, 'y': yf_te,
            'val_probs': v_probs, 'val_y': yf_v
        })

    avg_probs = np.mean([r['probs'] for r in seed_results], axis=0)
    avg_val_probs = np.mean([r['val_probs'] for r in seed_results], axis=0)
    ens_thr, _ = find_best_threshold(avg_val_probs[:, 1], yf_v)
    ens_preds = (avg_probs[:, 1] > ens_thr).astype(int)
    if detect_collapse(ens_preds):
        ens_preds = (avg_probs[:, 1] > 0.5).astype(int)
        ens_thr = 0.5
    ens_acc = accuracy_score(yf_te, ens_preds)
    ens_bal = balanced_accuracy_score(yf_te, ens_preds)
    ens_mcc = matthews_corrcoef(yf_te, ens_preds)
    try: ens_auc = roc_auc_score(yf_te, avg_probs[:, 1])
    except: ens_auc = 0.5
    print(f"  ENSEMBLE: thr={ens_thr:.2f} acc={ens_acc:.4f} bal={ens_bal:.4f} mcc={ens_mcc:+.3f} auc={ens_auc:.4f}")

    np.savez(f'{OUT_DIR}/XGBoost_{horizon}d_preds.npz',
             probs=avg_probs, preds=ens_preds, y=yf_te,
             val_probs=avg_val_probs, val_y=yf_v, threshold=ens_thr)

    return {'model': 'XGBoost', 'horizon': f'{horizon}d',
            'acc_mean': float(np.mean([r['acc'] for r in seed_results])),
            'acc_std': float(np.std([r['acc'] for r in seed_results])),
            'bal_acc_mean': float(np.mean([r['bal_acc'] for r in seed_results])),
            'mcc_mean': float(np.mean([r['mcc'] for r in seed_results])),
            'auc_mean': float(np.mean([r['auc'] for r in seed_results])),
            'ensemble_acc': float(ens_acc),
            'ensemble_bal_acc': float(ens_bal),
            'ensemble_mcc': float(ens_mcc),
            'ensemble_auc': float(ens_auc),
            'ensemble_threshold': float(ens_thr),
            'n_valid_seeds': len(seed_results)}

data1 = np.load('outputs/sequences/sequences_1d.npz', allow_pickle=True)
n_hard = data1['X_hard_train'].shape[-1]
n_soft = data1['X_soft_train'].shape[-1]
print(f"\nn_hard={n_hard} n_soft={n_soft}")

t_start = time.time()
results = []
for h in [1, 7, 30]:
    print(f"\n{'#'*70}\n# HORIZON: {h}d\n{'#'*70}")
    results.append(train_one(lambda: AttentionLSTM(n_hard), 'AttentionLSTM', False, h))
    results.append(train_one(lambda: TransformerPrice(n_hard), 'TransformerPrice', False, h))
    results.append(train_one(lambda: CrossAttentionFusion(n_hard, n_soft), 'CrossAttentionFusion', True, h))
    results.append(train_one(lambda: TCN(n_hard), 'TCN', False, h))
    results.append(train_xgb(h))

with open('outputs/models/training_summary.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*70}\nDone in {(time.time()-t_start)/60:.1f} min\n{'='*70}")
print(f"\n{'Model':<22}{'Hor':<5}{'Acc':<8}{'BalAcc':<9}{'MCC':<9}{'AUC':<8}{'Thr':<6}{'#OK':<5}")
print("-"*70)
for h in ['1d', '7d', '30d']:
    for r in sorted([x for x in results if x['horizon']==h], key=lambda x: -x['ensemble_auc']):
        print(f"{r['model']:<22}{r['horizon']:<5}{r['ensemble_acc']:<8.4f}{r['ensemble_bal_acc']:<9.4f}{r['ensemble_mcc']:<+9.3f}{r['ensemble_auc']:<8.4f}{r['ensemble_threshold']:<6.2f}{r['n_valid_seeds']:<5}")

Device: cuda
Config: epochs=80 batch=64 lr=0.0005 wd=0.005 patience=12

n_hard=70 n_soft=18

######################################################################
# HORIZON: 1d
######################################################################

AttentionLSTM | 1d
  seed=42 ep=  0 val_auc=0.5076 best=0.5076
  seed=42 ep=  3 val_auc=0.5113 best=0.5113
  seed=42 ep= 10 val_auc=0.4829 best=0.5113
  seed=42 thr=0.48 acc=0.5193 bal=0.5173 mcc=+0.035 auc=0.5152 pred_up%=54
  seed=123 ep=  0 val_auc=0.4796 best=0.4796
  seed=123 ep=  1 val_auc=0.4867 best=0.4867
  seed=123 ep=  3 val_auc=0.5017 best=0.5017
  seed=123 ep=  4 val_auc=0.5035 best=0.5035
  seed=123 ep=  5 val_auc=0.5192 best=0.5192
  seed=123 ep= 10 val_auc=0.5107 best=0.5192
  seed=123 ep= 14 val_auc=0.5209 best=0.5209
  seed=123 ep= 17 val_auc=0.5320 best=0.5320
  seed=123 ep= 20 val_auc=0.5320 best=0.5320
  seed=123 ep= 24 val_auc=0.5328 best=0.5328
  seed=123 ep= 26 val_auc=0.5331 best=0.5331
  seed=123 ep= 28 val_auc=0.5

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()*(-math.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos*div); pe[:, 1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class GaussianNoise(nn.Module):
    def __init__(self, sigma=0.05):
        super().__init__()
        self.sigma = sigma
    def forward(self, x):
        if self.training and self.sigma > 0:
            return x + torch.randn_like(x) * self.sigma
        return x

class AttentionLSTM(nn.Module):
    def __init__(self, n_features, hidden=64, n_layers=2, n_classes=2, dropout=0.5):
        super().__init__()
        self.input_noise = GaussianNoise(0.05)
        self.input_drop = nn.Dropout(0.2)
        self.lstm = nn.LSTM(n_features, hidden, n_layers, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.attn = nn.Sequential(nn.Linear(hidden*2, hidden), nn.Tanh(),
                                   nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.fc = nn.Sequential(
            nn.LayerNorm(hidden*2), nn.Dropout(dropout),
            nn.Linear(hidden*2, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, n_classes))
    def forward(self, x):
        x = self.input_noise(self.input_drop(x))
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (out*w).sum(dim=1)
        return self.fc(ctx), w.squeeze(-1)

class TransformerPrice(nn.Module):
    def __init__(self, n_features, d_model=64, nhead=4, n_layers=2, n_classes=2, dropout=0.4):
        super().__init__()
        self.input_noise = GaussianNoise(0.05)
        self.input_drop = nn.Dropout(0.2)
        self.proj = nn.Linear(n_features, d_model)
        self.pe = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model*2, dropout,
                                            batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.pool = nn.Linear(d_model, 1)
        self.fc = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout),
            nn.Linear(d_model, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, n_classes))
    def forward(self, x):
        x = self.input_noise(self.input_drop(x))
        h = self.pe(self.proj(x))
        h = self.encoder(h)
        w = torch.softmax(self.pool(h), dim=1)
        ctx = (h*w).sum(dim=1)
        return self.fc(ctx), w.squeeze(-1)

class CrossAttentionFusion(nn.Module):
    def __init__(self, n_hard, n_soft, d_model=64, nhead=4, n_layers=2, n_classes=2, dropout=0.4):
        super().__init__()
        self.hard_noise = GaussianNoise(0.05)
        self.soft_noise = GaussianNoise(0.05)
        self.hard_drop = nn.Dropout(0.2)
        self.soft_drop = nn.Dropout(0.2)
        self.hard_proj = nn.Linear(n_hard, d_model)
        self.soft_proj = nn.Linear(n_soft, d_model)
        self.pe_hard = PositionalEncoding(d_model)
        self.pe_soft = PositionalEncoding(d_model)
        hl = nn.TransformerEncoderLayer(d_model, nhead, d_model*2, dropout, batch_first=True, activation='gelu')
        sl = nn.TransformerEncoderLayer(d_model, nhead, d_model*2, dropout, batch_first=True, activation='gelu')
        self.hard_encoder = nn.TransformerEncoder(hl, n_layers)
        self.soft_encoder = nn.TransformerEncoder(sl, n_layers)
        self.cross = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.pool_h = nn.Linear(d_model, 1)
        self.pool_s = nn.Linear(d_model, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model*2), nn.Dropout(dropout),
            nn.Linear(d_model*2, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, n_classes))
    def forward(self, x_hard, x_soft):
        x_hard = self.hard_noise(self.hard_drop(x_hard))
        x_soft = self.soft_noise(self.soft_drop(x_soft))
        H = self.pe_hard(self.hard_proj(x_hard))
        S = self.pe_soft(self.soft_proj(x_soft))
        H = self.hard_encoder(H); S = self.soft_encoder(S)
        H_c, _ = self.cross(H, S, S, need_weights=False)
        H = self.norm(H + H_c)
        wh = torch.softmax(self.pool_h(H), dim=1)
        ws = torch.softmax(self.pool_s(S), dim=1)
        zh = (H*wh).sum(dim=1); zs = (S*ws).sum(dim=1)
        return self.classifier(torch.cat([zh, zs], dim=-1)), {'h':wh.squeeze(-1),'s':ws.squeeze(-1)}

class TCN(nn.Module):
    def __init__(self, n_features, channels=64, n_classes=2, dropout=0.4):
        super().__init__()
        self.input_noise = GaussianNoise(0.05)
        self.input_drop = nn.Dropout(0.2)
        layers = []
        in_ch = n_features
        for d in [1, 2, 4, 8]:
            layers += [
                nn.Conv1d(in_ch, channels, 3, padding=d, dilation=d),
                nn.BatchNorm1d(channels), nn.GELU(), nn.Dropout(dropout)
            ]
            in_ch = channels
        self.tcn = nn.Sequential(*layers)
        self.attn = nn.Sequential(nn.Linear(channels, channels), nn.Tanh(), nn.Linear(channels, 1))
        self.fc = nn.Sequential(
            nn.LayerNorm(channels), nn.Dropout(dropout),
            nn.Linear(channels, channels), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(channels, n_classes))
    def forward(self, x):
        x = self.input_noise(self.input_drop(x))
        h = self.tcn(x.transpose(1, 2)).transpose(1, 2)
        w = torch.softmax(self.attn(h), dim=1)
        ctx = (h*w).sum(dim=1)
        return self.fc(ctx), w.squeeze(-1)

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5, label_smoothing=0.1):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma; self.ls=label_smoothing
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha,
                             label_smoothing=self.ls, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

print("Models defined: smaller capacity, dropout 0.4-0.5, input noise, layernorm")

Models defined: smaller capacity, dropout 0.4-0.5, input noise, layernorm


In [ ]:
import numpy as np, torch, os, json, random, time, math
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score, balanced_accuracy_score

OUT_DIR = 'outputs/models'
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 80
BATCH = 64
LR = 5e-4
WD = 5e-3
PATIENCE = 12
SEEDS = [42, 123, 2024, 7, 999]

print(f"Device: {DEVICE}")
print(f"Config: epochs={EPOCHS} batch={BATCH} lr={LR} wd={WD} patience={PATIENCE}")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_probs(model, loader, use_soft):
    model.eval()
    all_pr, all_y = [], []
    with torch.no_grad():
        for batch in loader:
            if use_soft:
                xh, xs, y = batch
                xh, xs = xh.to(DEVICE), xs.to(DEVICE)
                out, _ = model(xh, xs)
            else:
                xh, y = batch
                xh = xh.to(DEVICE)
                out, _ = model(xh)
            pr = torch.softmax(out, dim=-1).cpu().numpy()
            all_pr.append(pr); all_y.append(y.numpy())
    return np.concatenate(all_pr), np.concatenate(all_y)

def find_best_threshold_v2(probs_up, y_true, min_pred_rate=0.20, max_pred_rate=0.80):
    best_thr, best_score = 0.5, -1
    for thr in np.arange(0.20, 0.81, 0.01):
        preds = (probs_up > thr).astype(int)
        pred_up_rate = preds.mean()
        if pred_up_rate < min_pred_rate or pred_up_rate > max_pred_rate:
            continue
        bal = balanced_accuracy_score(y_true, preds)
        try:
            mcc_val = matthews_corrcoef(y_true, preds)
        except:
            mcc_val = 0
        score = bal + 0.3 * mcc_val
        if score > best_score:
            best_score = score
            best_thr = thr

    if best_score < 0:
        median_thr = np.median(probs_up)
        return float(median_thr), 0.5

    return best_thr, best_score

def detect_collapse(preds):
    if len(preds) == 0:
        return True
    rate1 = (preds == 1).mean()
    return rate1 < 0.10 or rate1 > 0.90

def train_one(model_fn, name, use_soft, horizon):
    print(f"\n{'='*70}\n{name} | {horizon}d\n{'='*70}")
    data = np.load(f'outputs/sequences/sequences_{horizon}d.npz', allow_pickle=True)
    Xh_tr, Xs_tr, y_tr = data['X_hard_train'], data['X_soft_train'], data['y_train']
    Xh_v, Xs_v, y_v = data['X_hard_val'], data['X_soft_val'], data['y_val']
    Xh_te, Xs_te, y_te = data['X_hard_test'], data['X_soft_test'], data['y_test']

    cw = compute_class_weight('balanced', classes=np.array([0,1]), y=y_tr)
    cw_t = torch.tensor(cw, dtype=torch.float32).to(DEVICE)

    if use_soft:
        tr_ds = TensorDataset(torch.tensor(Xh_tr), torch.tensor(Xs_tr), torch.tensor(y_tr))
        v_ds = TensorDataset(torch.tensor(Xh_v), torch.tensor(Xs_v), torch.tensor(y_v))
        te_ds = TensorDataset(torch.tensor(Xh_te), torch.tensor(Xs_te), torch.tensor(y_te))
    else:
        tr_ds = TensorDataset(torch.tensor(Xh_tr), torch.tensor(y_tr))
        v_ds = TensorDataset(torch.tensor(Xh_v), torch.tensor(y_v))
        te_ds = TensorDataset(torch.tensor(Xh_te), torch.tensor(y_te))

    tr_l = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, drop_last=True)
    v_l = DataLoader(v_ds, batch_size=BATCH, shuffle=False)
    te_l = DataLoader(te_ds, batch_size=BATCH, shuffle=False)

    seed_results = []
    for seed in SEEDS:
        set_seed(seed)
        model = model_fn().to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        warmup = 5
        def ll(ep):
            if ep < warmup: return (ep+1) / warmup
            prog = (ep - warmup) / max(1, EPOCHS - warmup)
            return 0.5 * (1 + math.cos(math.pi * prog))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, ll)
        crit = FocalLoss(alpha=cw_t, gamma=1.5, label_smoothing=0.1)

        best_auc, best_state, pat = -1, None, 0
        for ep in range(EPOCHS):
            model.train()
            for batch in tr_l:
                if use_soft:
                    xh, xs, y = batch
                    xh, xs, y = xh.to(DEVICE), xs.to(DEVICE), y.to(DEVICE)
                    out, _ = model(xh, xs)
                else:
                    xh, y = batch
                    xh, y = xh.to(DEVICE), y.to(DEVICE)
                    out, _ = model(xh)
                loss = crit(out, y)
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                opt.step()
            sched.step()

            v_pr, v_y = get_probs(model, v_l, use_soft)
            try: v_auc = roc_auc_score(v_y, v_pr[:, 1])
            except: v_auc = 0.5

            if v_auc > best_auc:
                best_auc = v_auc
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                pat = 0
            else:
                pat += 1

            if ep % 10 == 0 or pat == 0:
                print(f"  seed={seed} ep={ep:3d} val_auc={v_auc:.4f} best={best_auc:.4f}")

            if pat >= PATIENCE:
                break

        model.load_state_dict(best_state)

        v_pr, v_y = get_probs(model, v_l, use_soft)
        te_pr, te_y = get_probs(model, te_l, use_soft)

        thr_v2, _ = find_best_threshold_v2(v_pr[:, 1], v_y)
        te_preds = (te_pr[:, 1] > thr_v2).astype(int)

        if detect_collapse(te_preds):
            target_up_rate = max(0.30, min(0.70, v_y.mean()))
            target_thr = np.quantile(te_pr[:, 1], 1 - target_up_rate)
            te_preds = (te_pr[:, 1] > target_thr).astype(int)
            thr_v2 = target_thr

        ta = accuracy_score(te_y, te_preds)
        tba = balanced_accuracy_score(te_y, te_preds)
        tf = f1_score(te_y, te_preds, average='macro', zero_division=0)
        tm = matthews_corrcoef(te_y, te_preds)
        try: tauc = roc_auc_score(te_y, te_pr[:, 1])
        except: tauc = 0.5

        print(f"  seed={seed} thr={thr_v2:.3f} acc={ta:.4f} bal={tba:.4f} mcc={tm:+.3f} auc={tauc:.4f} pred_up%={100*(te_preds==1).mean():.0f}")

        seed_results.append({
            'seed': seed, 'acc': ta, 'bal_acc': tba, 'f1': tf, 'mcc': tm, 'auc': tauc,
            'thr': thr_v2, 'probs': te_pr, 'preds': te_preds, 'y': te_y,
            'val_probs': v_pr, 'val_y': v_y, 'state': best_state
        })

    valid = [r for r in seed_results if not detect_collapse(r['preds'])]
    if len(valid) == 0:
        print(f"  All {len(seed_results)} seeds collapsed. Selecting top-3 by AUC for ensemble.")
        seed_results.sort(key=lambda x: -x['auc'])
        valid = seed_results[:3]

    avg_probs = np.mean([r['probs'] for r in valid], axis=0)
    avg_val_probs = np.mean([r['val_probs'] for r in valid], axis=0)

    ens_thr, _ = find_best_threshold_v2(avg_val_probs[:, 1], valid[0]['val_y'])
    ens_preds = (avg_probs[:, 1] > ens_thr).astype(int)

    if detect_collapse(ens_preds):
        target_up_rate = max(0.35, min(0.65, valid[0]['val_y'].mean()))
        ens_thr = np.quantile(avg_probs[:, 1], 1 - target_up_rate)
        ens_preds = (avg_probs[:, 1] > ens_thr).astype(int)
        print(f"  Ensemble forced to median split: thr={ens_thr:.3f}")

    ens_y = valid[0]['y']
    ens_acc = accuracy_score(ens_y, ens_preds)
    ens_bal = balanced_accuracy_score(ens_y, ens_preds)
    ens_mcc = matthews_corrcoef(ens_y, ens_preds)
    try: ens_auc = roc_auc_score(ens_y, avg_probs[:, 1])
    except: ens_auc = 0.5

    print(f"  ENSEMBLE: thr={ens_thr:.3f} acc={ens_acc:.4f} bal={ens_bal:.4f} mcc={ens_mcc:+.3f} auc={ens_auc:.4f} pred_up%={100*(ens_preds==1).mean():.0f}")

    key = f'{name}_{horizon}d'
    best_seed_state = max(seed_results, key=lambda x: x['auc'])['state']
    torch.save(best_seed_state, f'{OUT_DIR}/{key}.pt')
    np.savez(f'{OUT_DIR}/{key}_preds.npz',
             probs=avg_probs, preds=ens_preds, y=ens_y,
             val_probs=avg_val_probs, val_y=valid[0]['val_y'],
             threshold=ens_thr)

    return {'model': name, 'horizon': f'{horizon}d',
            'acc_mean': float(np.mean([r['acc'] for r in valid])),
            'acc_std': float(np.std([r['acc'] for r in valid])),
            'bal_acc_mean': float(np.mean([r['bal_acc'] for r in valid])),
            'mcc_mean': float(np.mean([r['mcc'] for r in valid])),
            'auc_mean': float(np.mean([r['auc'] for r in valid])),
            'ensemble_acc': float(ens_acc),
            'ensemble_bal_acc': float(ens_bal),
            'ensemble_mcc': float(ens_mcc),
            'ensemble_auc': float(ens_auc),
            'ensemble_threshold': float(ens_thr),
            'n_valid_seeds': len(valid)}

def train_xgb(horizon):
    import xgboost as xgb
    print(f"\n{'='*70}\nXGBoost | {horizon}d\n{'='*70}")
    data = np.load(f'outputs/sequences/sequences_{horizon}d.npz', allow_pickle=True)
    Xh_tr_seq, y_tr_seq = data['X_hard_train'], data['y_train']
    Xs_tr_seq = data['X_soft_train']
    Xh_v_seq, y_v_seq = data['X_hard_val'], data['y_val']
    Xs_v_seq = data['X_soft_val']
    Xh_te_seq, y_te_seq = data['X_hard_test'], data['y_test']
    Xs_te_seq = data['X_soft_test']

    def aggregate_sequences(Xh, Xs):
        feats = []
        last_hard = Xh[:, -1, :]
        last_soft = Xs[:, -1, :]
        mean_hard = Xh.mean(axis=1)
        mean_soft = Xs.mean(axis=1)
        std_hard_5 = Xh[:, -5:, :].std(axis=1)
        return np.concatenate([last_hard, last_soft, mean_hard, mean_soft, std_hard_5], axis=1)

    Xf_tr = aggregate_sequences(Xh_tr_seq, Xs_tr_seq)
    Xf_v = aggregate_sequences(Xh_v_seq, Xs_v_seq)
    Xf_te = aggregate_sequences(Xh_te_seq, Xs_te_seq)
    yf_tr = y_tr_seq; yf_v = y_v_seq; yf_te = y_te_seq

    print(f"  XGB feature dim: {Xf_tr.shape[1]} (last + mean + 5d-std aggregations)")

    sw = compute_sample_weight('balanced', yf_tr)
    seed_results = []
    for seed in SEEDS:
        clf = xgb.XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.03,
            subsample=0.7, colsample_bytree=0.6,
            min_child_weight=10, gamma=0.5, reg_lambda=2.0, reg_alpha=0.5,
            objective='binary:logistic', eval_metric='auc',
            early_stopping_rounds=20, random_state=seed, n_jobs=-1, verbosity=0)
        clf.fit(Xf_tr, yf_tr, sample_weight=sw, eval_set=[(Xf_v, yf_v)], verbose=False)

        v_probs = clf.predict_proba(Xf_v)
        thr, _ = find_best_threshold_v2(v_probs[:, 1], yf_v)
        t_probs = clf.predict_proba(Xf_te)
        t_preds = (t_probs[:, 1] > thr).astype(int)
        if detect_collapse(t_preds):
            target = max(0.30, min(0.70, yf_v.mean()))
            thr = np.quantile(t_probs[:, 1], 1 - target)
            t_preds = (t_probs[:, 1] > thr).astype(int)
        acc = accuracy_score(yf_te, t_preds)
        bal = balanced_accuracy_score(yf_te, t_preds)
        mcc = matthews_corrcoef(yf_te, t_preds)
        try: auc = roc_auc_score(yf_te, t_probs[:, 1])
        except: auc = 0.5
        print(f"  seed={seed} thr={thr:.3f} acc={acc:.4f} bal={bal:.4f} mcc={mcc:+.3f} auc={auc:.4f}")
        seed_results.append({
            'seed': seed, 'acc': acc, 'bal_acc': bal, 'mcc': mcc, 'auc': auc,
            'thr': thr, 'probs': t_probs, 'preds': t_preds, 'y': yf_te,
            'val_probs': v_probs, 'val_y': yf_v
        })

    avg_probs = np.mean([r['probs'] for r in seed_results], axis=0)
    avg_val_probs = np.mean([r['val_probs'] for r in seed_results], axis=0)
    ens_thr, _ = find_best_threshold_v2(avg_val_probs[:, 1], yf_v)
    ens_preds = (avg_probs[:, 1] > ens_thr).astype(int)
    if detect_collapse(ens_preds):
        target = max(0.35, min(0.65, yf_v.mean()))
        ens_thr = np.quantile(avg_probs[:, 1], 1 - target)
        ens_preds = (avg_probs[:, 1] > ens_thr).astype(int)
    ens_acc = accuracy_score(yf_te, ens_preds)
    ens_bal = balanced_accuracy_score(yf_te, ens_preds)
    ens_mcc = matthews_corrcoef(yf_te, ens_preds)
    try: ens_auc = roc_auc_score(yf_te, avg_probs[:, 1])
    except: ens_auc = 0.5
    print(f"  ENSEMBLE: thr={ens_thr:.3f} acc={ens_acc:.4f} bal={ens_bal:.4f} mcc={ens_mcc:+.3f} auc={ens_auc:.4f}")

    np.savez(f'{OUT_DIR}/XGBoost_{horizon}d_preds.npz',
             probs=avg_probs, preds=ens_preds, y=yf_te,
             val_probs=avg_val_probs, val_y=yf_v, threshold=ens_thr)

    return {'model': 'XGBoost', 'horizon': f'{horizon}d',
            'acc_mean': float(np.mean([r['acc'] for r in seed_results])),
            'acc_std': float(np.std([r['acc'] for r in seed_results])),
            'bal_acc_mean': float(np.mean([r['bal_acc'] for r in seed_results])),
            'mcc_mean': float(np.mean([r['mcc'] for r in seed_results])),
            'auc_mean': float(np.mean([r['auc'] for r in seed_results])),
            'ensemble_acc': float(ens_acc),
            'ensemble_bal_acc': float(ens_bal),
            'ensemble_mcc': float(ens_mcc),
            'ensemble_auc': float(ens_auc),
            'ensemble_threshold': float(ens_thr),
            'n_valid_seeds': len(seed_results)}

data1 = np.load('outputs/sequences/sequences_1d.npz', allow_pickle=True)
n_hard = data1['X_hard_train'].shape[-1]
n_soft = data1['X_soft_train'].shape[-1]
print(f"\nn_hard={n_hard} n_soft={n_soft}")

t_start = time.time()
results = []
for h in [1, 7, 30]:
    print(f"\n{'#'*70}\n# HORIZON: {h}d\n{'#'*70}")
    results.append(train_one(lambda: AttentionLSTM(n_hard), 'AttentionLSTM', False, h))
    results.append(train_one(lambda: TransformerPrice(n_hard), 'TransformerPrice', False, h))
    results.append(train_one(lambda: CrossAttentionFusion(n_hard, n_soft), 'CrossAttentionFusion', True, h))
    results.append(train_one(lambda: TCN(n_hard), 'TCN', False, h))
    results.append(train_xgb(h))

with open('outputs/models/training_summary.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*70}\nDone in {(time.time()-t_start)/60:.1f} min\n{'='*70}")
print(f"\n{'Model':<22}{'Hor':<5}{'Acc':<8}{'BalAcc':<9}{'MCC':<9}{'AUC':<8}{'Thr':<7}{'#OK':<5}")
print("-"*73)
for h in ['1d', '7d', '30d']:
    for r in sorted([x for x in results if x['horizon']==h], key=lambda x: -x['ensemble_auc']):
        print(f"{r['model']:<22}{r['horizon']:<5}{r['ensemble_acc']:<8.4f}{r['ensemble_bal_acc']:<9.4f}{r['ensemble_mcc']:<+9.3f}{r['ensemble_auc']:<8.4f}{r['ensemble_threshold']:<7.3f}{r['n_valid_seeds']:<5}")

Device: cuda
Config: epochs=80 batch=64 lr=0.0005 wd=0.005 patience=12

n_hard=70 n_soft=18

######################################################################
# HORIZON: 1d
######################################################################

AttentionLSTM | 1d
  seed=42 ep=  0 val_auc=0.5076 best=0.5076
  seed=42 ep=  3 val_auc=0.5113 best=0.5113
  seed=42 ep= 10 val_auc=0.4829 best=0.5113
  seed=42 thr=0.480 acc=0.5193 bal=0.5173 mcc=+0.035 auc=0.5152 pred_up%=54
  seed=123 ep=  0 val_auc=0.4796 best=0.4796
  seed=123 ep=  1 val_auc=0.4867 best=0.4867
  seed=123 ep=  3 val_auc=0.5017 best=0.5017
  seed=123 ep=  4 val_auc=0.5035 best=0.5035
  seed=123 ep=  5 val_auc=0.5192 best=0.5192
  seed=123 ep= 10 val_auc=0.5107 best=0.5192
  seed=123 ep= 14 val_auc=0.5209 best=0.5209
  seed=123 ep= 17 val_auc=0.5320 best=0.5320
  seed=123 ep= 20 val_auc=0.5320 best=0.5320
  seed=123 ep= 24 val_auc=0.5328 best=0.5328
  seed=123 ep= 26 val_auc=0.5331 best=0.5331
  seed=123 ep= 28 val_auc=0.

In [ ]:
import numpy as np, pandas as pd, os, json, shutil
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression

OUT_DIR = 'outputs/evaluation'
os.makedirs(OUT_DIR, exist_ok=True)

def detect_collapse(preds):
    if len(preds) == 0: return True
    rate1 = (preds == 1).mean()
    return rate1 < 0.05 or rate1 > 0.95

def financial_metrics(preds, returns, confidence, threshold, horizon):
    preds = np.asarray(preds); returns = np.asarray(returns)
    active = confidence > threshold
    position = np.zeros_like(preds, dtype=float)
    position[(preds == 1) & active] = 1.0
    position[(preds == 0) & active] = -1.0
    trade_idx = np.arange(0, len(preds), max(1, horizon // 2))
    tp = position[trade_idx]; tr = returns[trade_idx]
    gr = tp * tr; cost = 0.0015 * np.abs(tp)
    net = np.clip(gr - cost, -0.95, 3.0)
    if len(net) == 0:
        return {'final_return': 0, 'sharpe': 0, 'max_dd': 0, 'n_trades': 0, 'win_rate': 0}
    cumret = np.cumprod(1 + net) - 1
    fr = cumret[-1]
    sharpe = np.sqrt(252/horizon) * net.mean() / (net.std() + 1e-10) if net.std() > 0 else 0
    eq = np.cumprod(1 + net); peak = np.maximum.accumulate(eq)
    dd = (eq - peak) / peak
    max_dd = dd.min() if len(dd) else 0
    tm = tp != 0; win = (gr[tm] > 0).mean() if tm.sum() > 0 else 0
    return {'final_return': float(fr), 'sharpe': float(sharpe), 'max_dd': float(max_dd),
            'n_trades': int(tm.sum()), 'win_rate': float(win)}

def evaluate(name, horizon, probs, preds, y, returns):
    acc = accuracy_score(y, preds)
    bal = balanced_accuracy_score(y, preds)
    f1 = f1_score(y, preds, average='macro', zero_division=0)
    mcc = matthews_corrcoef(y, preds)
    try: auc = roc_auc_score(y, probs[:,1])
    except: auc = 0.5
    pred_up = 100*(preds==1).mean()
    collapsed = detect_collapse(preds)

    print(f"\n{name} | {horizon}: acc={acc:.4f} bal={bal:.4f} f1={f1:.4f} mcc={mcc:+.3f} auc={auc:.4f} up%={pred_up:.0f} collapsed={collapsed}")

    res = {'model': name, 'horizon': horizon,
           'acc': float(acc), 'bal_acc': float(bal), 'f1': float(f1),
           'mcc': float(mcc), 'auc': float(auc), 'pred_up_rate': float(pred_up),
           'collapsed': bool(collapsed), 'hc_acc': {}, 'strategies': {}}

    conf = probs.max(axis=1)
    for thr in [0.55, 0.60, 0.65, 0.70, 0.75]:
        mask = conf > thr
        if mask.sum() > 5:
            sub_preds = preds[mask]
            if detect_collapse(sub_preds):
                continue
            hc = accuracy_score(y[mask], sub_preds)
            print(f"  Conf>{thr}: bal={balanced_accuracy_score(y[mask], sub_preds):.4f} acc={hc:.4f} (n={mask.sum()}, {100*mask.mean():.1f}%)")
            res['hc_acc'][str(thr)] = {
                'acc': float(hc),
                'bal_acc': float(balanced_accuracy_score(y[mask], sub_preds)),
                'n': int(mask.sum()), 'coverage': float(mask.mean())}

    h_num = int(horizon.replace('d',''))
    for lbl, t in [('all', 0.0), ('c060', 0.60), ('c070', 0.70)]:
        fm = financial_metrics(preds, returns, conf, t, h_num)
        print(f"  [{lbl:5s}] ret={fm['final_return']*100:+.1f}% sharpe={fm['sharpe']:+.2f} trades={fm['n_trades']} win={fm['win_rate']:.1%}")
        res['strategies'][lbl] = fm
    return res

models = ['AttentionLSTM','TransformerPrice','CrossAttentionFusion','TCN','XGBoost']
horizons = ['1d','7d','30d']
all_results = []

for h in horizons:
    print(f"\n{'='*80}\nHORIZON {h}\n{'='*80}")
    data = np.load(f'outputs/sequences/sequences_{h}.npz', allow_pickle=True)
    r_te = data['r_test']; r_te_f = data['r_flat_test']

    model_probs = {}; model_val_probs = {}
    y_test_ref = None; y_val_ref = None

    for m in models:
        p = f'outputs/models/{m}_{h}_preds.npz'
        if not os.path.exists(p): continue
        d = np.load(p)
        probs = d['probs']; preds = d['preds']; y = d['y']
        rets = r_te_f if m == 'XGBoost' else r_te
        n = min(len(probs), len(rets))
        res = evaluate(m, h, probs[:n], preds[:n], y[:n], rets[:n])
        all_results.append(res)
        if m != 'XGBoost':
            model_probs[m] = probs[:n]
            if 'val_probs' in d.files:
                model_val_probs[m] = d['val_probs']
                y_val_ref = d['val_y']
            if y_test_ref is None:
                y_test_ref = y[:n]

    if len(model_probs) >= 3 and len(model_val_probs) >= 3:
        seq_ms = list(model_val_probs.keys())
        min_val = min(len(model_val_probs[m]) for m in seq_ms)
        min_val = min(min_val, len(y_val_ref))

        X_meta_v = np.column_stack([model_val_probs[m][:min_val,1] for m in seq_ms])
        meta = LogisticRegression(C=0.1, max_iter=1000, random_state=42, class_weight='balanced')
        meta.fit(X_meta_v, y_val_ref[:min_val])

        v_meta_probs = meta.predict_proba(X_meta_v)
        best_thr_meta, best_score_meta = 0.5, -1
        for thr in np.arange(0.30, 0.71, 0.01):
            preds_v = (v_meta_probs[:, 1] > thr).astype(int)
            if preds_v.sum() == 0 or preds_v.sum() == len(preds_v): continue
            score = balanced_accuracy_score(y_val_ref[:min_val], preds_v)
            if score > best_score_meta:
                best_score_meta = score; best_thr_meta = thr

        X_meta_t = np.column_stack([model_probs[m][:,1] for m in seq_ms])
        s_up = meta.predict_proba(X_meta_t)[:,1]
        s_probs = np.column_stack([1-s_up, s_up])
        s_preds = (s_up > best_thr_meta).astype(int)
        if detect_collapse(s_preds):
            s_preds = (s_up > 0.5).astype(int)

        all_results.append(evaluate('StackedEnsemble', h, s_probs, s_preds, y_test_ref, r_te[:len(y_test_ref)]))

        avg_up = np.mean([model_probs[m][:,1] for m in seq_ms], axis=0)
        avg_val_up = np.mean([model_val_probs[m][:min_val,1] for m in seq_ms], axis=0)
        best_thr_avg = 0.5
        best_score_avg = -1
        for thr in np.arange(0.30, 0.71, 0.01):
            preds_v = (avg_val_up > thr).astype(int)
            if preds_v.sum() == 0 or preds_v.sum() == len(preds_v): continue
            score = balanced_accuracy_score(y_val_ref[:min_val], preds_v)
            if score > best_score_avg:
                best_score_avg = score; best_thr_avg = thr
        a_probs = np.column_stack([1-avg_up, avg_up])
        a_preds = (avg_up > best_thr_avg).astype(int)
        if detect_collapse(a_preds):
            a_preds = (avg_up > 0.5).astype(int)
        all_results.append(evaluate('AvgEnsemble', h, a_probs, a_preds, y_test_ref, r_te[:len(y_test_ref)]))

with open(f'{OUT_DIR}/evaluation_summary.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"\n{'='*120}")
print(f"FINAL SUMMARY")
print(f"{'='*120}")
print(f"{'Model':<22}{'Hor':<5}{'Acc':<8}{'BalAcc':<9}{'F1':<8}{'MCC':<9}{'AUC':<8}{'Up%':<6}{'Status':<10}")
print("-"*120)

for h in horizons:
    h_results = [r for r in all_results if r['horizon'] == h]
    h_results.sort(key=lambda x: -x['auc'])
    for r in h_results:
        status = "COLLAPSED" if r['collapsed'] else ("WEAK" if r['mcc'] < 0.05 else "GOOD")
        print(f"{r['model']:<22}{r['horizon']:<5}{r['acc']:<8.4f}{r['bal_acc']:<9.4f}{r['f1']:<8.4f}{r['mcc']:<+9.3f}{r['auc']:<8.4f}{r['pred_up_rate']:<6.0f}{status:<10}")
    print()

DRIVE_OUT = f'{DRIVE_DIR}/{COIN_NAME.lower()}_v2_outputs'
os.makedirs(DRIVE_OUT, exist_ok=True)
if os.path.exists(f'{DRIVE_OUT}/outputs'):
    shutil.rmtree(f'{DRIVE_OUT}/outputs')
shutil.copytree('outputs', f'{DRIVE_OUT}/outputs')
print(f"Backup: {DRIVE_OUT}")


HORIZON 1d

AttentionLSTM | 1d: acc=0.5193 bal=0.5245 f1=0.5164 mcc=+0.050 auc=0.5220 up%=40 collapsed=False
  [all  ] ret=-48.0% sharpe=-1.35 trades=181 win=51.9%
  [c060 ] ret=+0.0% sharpe=+0.00 trades=0 win=0.0%
  [c070 ] ret=+0.0% sharpe=+0.00 trades=0 win=0.0%

TransformerPrice | 1d: acc=0.4862 bal=0.4912 f1=0.4831 mcc=-0.018 auc=0.5220 up%=40 collapsed=False
  Conf>0.55: bal=0.4977 acc=0.4769 (n=130, 71.8%)
  [all  ] ret=-35.4% sharpe=-0.82 trades=181 win=48.6%
  [c060 ] ret=+0.0% sharpe=+0.00 trades=0 win=0.0%
  [c070 ] ret=+0.0% sharpe=+0.00 trades=0 win=0.0%

CrossAttentionFusion | 1d: acc=0.5470 bal=0.5288 f1=0.4646 mcc=+0.085 auc=0.5701 up%=87 collapsed=False
  [all  ] ret=+8.2% sharpe=+0.47 trades=181 win=54.7%
  [c060 ] ret=+0.0% sharpe=+0.00 trades=0 win=0.0%
  [c070 ] ret=+0.0% sharpe=+0.00 trades=0 win=0.0%

TCN | 1d: acc=0.5138 bal=0.5192 f1=0.5104 mcc=+0.039 auc=0.5310 up%=39 collapsed=False
  Conf>0.55: bal=0.4913 acc=0.4892 (n=139, 76.8%)
  [all  ] ret=-32.3% sharp